In [ ]:
# sinh con , check goal, insert queue
import tkinter as tk
from collections import deque
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300
def neighbors(state):
    idx = state.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(state)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result
def bfs_v1(start):
    if start == GOAL:
        return [start], 0
    visited = {start}
    queue = deque([(start, [start])])
    nodes = 0
    while queue:
        state, path = queue.popleft()
        nodes += 1
        for child in neighbors(state):
            if child in visited:
                continue
            if child == GOAL:  # check goal
                return path + [child], nodes
            visited.add(child)
            queue.append((child, path + [child]))
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0
def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t
class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle BFS")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.playing = False
        self.canvas = tk.Canvas(
            root,
            width=3*TILE + 4*GAP,
            height=3*TILE + 4*GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack(pady=10)
        info = tk.Frame(root, bg="white")
        info.pack()
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=10)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=10)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=10)
        btns = tk.Frame(root, bg="white")
        btns.pack(pady=10)
        tk.Button(
            btns,
            text="Shuffle",
            width=10,
            command=self.shuffle
        ).grid(row=0, column=0, padx=5)

        tk.Button(
            btns,
            text="Solve",
            width=10,
            command=self.solve
        ).grid(row=0, column=1, padx=5)

        tk.Button(
            btns,
            text="Step",
            width=10,
            command=self.next_step
        ).grid(row=0, column=2, padx=5)

        self.draw()
    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board
    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE
            self.canvas.create_rectangle(
                x1, y1, x2, y2,
                fill="white",
                outline="black",
                width=2
            )
            if val != 0:
                self.canvas.create_text(
                    (x1+x2)//2,
                    (y1+y2)//2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="black"
                )
    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.draw()
    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = bfs_v1(self.board)
        ms = (time.perf_counter() - t0) * 1000
        if path is None:
            return
        self.solution = path
        self.step = 0
        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path)-1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.playing = True
        self.animate()
    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
#sinh con ,insert queue, pop ra, check goal
import tkinter as tk
from collections import deque
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300
def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def bfs_v2(start):
    reached = {start}
    queue = deque([(start, [start])])
    nodes = 0
    while queue:
        board, path = queue.popleft()
        nodes += 1
        if board == GOAL: # pop ra roi check goal
            return path, nodes
        for child in get_neighbors(board):
            if child not in reached:
                reached.add(child)
                queue.append((child, path + [child])) # insert queue
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle BFS V2")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.playing = False
        self.canvas = tk.Canvas(
            root,
            width=3*TILE + 4*GAP,
            height=3*TILE + 4*GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack(pady=10)
        info = tk.Frame(root, bg="white")
        info.pack()
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=10)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=10)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=10)
        btns = tk.Frame(root, bg="white")
        btns.pack(pady=10)

        tk.Button(
            btns,
            text="Shuffle",
            width=10,
            command=self.shuffle
        ).grid(row=0, column=0, padx=5)

        tk.Button(
            btns,
            text="Reset",
            width=10,
            command=self.reset
        ).grid(row=0, column=1, padx=5)

        tk.Button(
            btns,
            text="Solve",
            width=10,
            command=self.solve
        ).grid(row=0, column=2, padx=5)

        tk.Button(
            btns,
            text="Step",
            width=34,
            command=self.next_step
        ).grid(row=1, column=0, columnspan=3, pady=5)
        self.draw()

    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE
            self.canvas.create_rectangle(
                x1,
                y1,
                x2,
                y2,
                fill="white",
                outline="black",
                width=2
            )
            if val != 0:
                self.canvas.create_text(
                    (x1+x2)//2,
                    (y1+y2)//2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="black"
                )
    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.step = 0
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.draw()

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = bfs_v2(self.board)
        ms = (time.perf_counter() - t0) * 1000
        if path is None:
            return
        self.solution = path
        self.step = 0
        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path)-1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.playing = True
        self.animate()

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
def get_children(state):
    idx = state.index(0)
    r, c = divmod(idx, 3)
    children = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            lst = list(state)
            lst[idx], lst[ni] = lst[ni], lst[idx]
            children.append(tuple(lst))
    return children


def get_move(a, b):
    ia = a.index(0)
    ib = b.index(0)
    diff = ib - ia
    if diff == -3:
        return "UP"
    if diff == 3:
        return "DOWN"
    if diff == -1:
        return "LEFT"
    return "RIGHT"


def dfs(start, limit=30):
    if start == GOAL:
        return [start], 0
    stack = [(start, [start], {start})]
    nodes = 0
    while stack:
        state, path, visited = stack.pop()
        nodes += 1
        if len(path) - 1 >= limit:
            continue
        for child in get_children(state):
            if child in visited:
                continue
            if child == GOAL:
                return path + [child], nodes
            stack.append(
                (
                    child,
                    path + [child],
                    visited | {child}
                )
            )
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):

            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0


def random_board():
    board = list(GOAL)
    while True:
        random.shuffle(board)
        s = tuple(board)
        if is_solvable(s) and s != GOAL:
            return s
class App:

    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle DFS")
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        main = tk.Frame(root)
        main.pack(padx=10, pady=10)
        left = tk.Frame(main)
        left.pack(side="left", padx=10)
        board_frame = tk.Frame(left)
        board_frame.pack()
        self.tiles = []
        for i in range(9):
            lbl = tk.Label(
                board_frame,
                width=4,
                height=2,
                font=("Arial", 24),
                relief="solid",
                borderwidth=1
            )

            lbl.grid(
                row=i//3,
                column=i%3
            )

            self.tiles.append(lbl)

        self.step_label = tk.Label(
            left,
            text="Steps:",
            font=("Arial", 12)
        )

        self.step_label.pack(pady=10)
        limit_frame = tk.Frame(left)
        limit_frame.pack(pady=5)

        tk.Label(
            limit_frame,
            text="Depth Limit:"
        ).pack(side="left")

        self.limit_entry = tk.Entry(
            limit_frame,
            width=5
        )

        self.limit_entry.insert(0, "30")

        self.limit_entry.pack(side="left", padx=5)

        btn = tk.Frame(left)
        btn.pack(pady=5)

        tk.Button(
            btn,
            text="Shuffle",
            width=10,
            command=self.shuffle
        ).pack(side="left", padx=5)

        tk.Button(
            btn,
            text="Solve",
            width=10,
            command=self.solve
        ).pack(side="left", padx=5)

        right = tk.Frame(main)
        right.pack(side="right")

        tk.Label(
            right,
            text="Solutions",
            font=("Arial", 14, "bold")
        ).pack()

        self.solution_box = tk.Text(
            right,
            width=25,
            height=20,
            font=("Consolas", 10)
        )

        self.solution_box.pack()
        self.info = tk.Label(root, text="")
        self.info.pack(pady=5)
        self.draw(self.board)

    def draw(self, board):
        for i, n in enumerate(board):
            self.tiles[i]["text"] = "" if n == 0 else str(n)
    def shuffle(self):
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.step_label["text"] = "Steps:"
        self.info["text"] = ""
        self.draw(self.board)
    def solve(self):
        limit = int(self.limit_entry.get())
        t0 = time.time()
        path, nodes = dfs(
            self.board,
            limit
        )
        elapsed = (time.time() - t0) * 1000
        if not path:
            self.info["text"] = (
                f"No solution within depth {limit}"
            )
            return
        self.solution = path
        self.moves = []
        for i in range(len(path)-1):
            move = get_move(
                path[i],
                path[i+1]
            )
            self.moves.append(move)
        self.step = 0

        self.info["text"] = (
            f"Nodes: {nodes}   "
            f"Steps: {len(path)-1}   "
            f"Time: {elapsed:.2f} ms"
        )
        self.solution_box.delete("1.0", tk.END)
        for i, m in enumerate(self.moves, 1):
            self.solution_box.insert(
                tk.END,
                f"{i}. {m}\n"
            )
        self.animate()
    def animate(self):

        if self.step >= len(self.solution):
            return
        self.draw(self.solution[self.step])
        if self.step == 0:
            self.step_label["text"] = "START"

        else:
            self.step_label["text"] = (
                f"Step {self.step}: "
                f"{self.moves[self.step-1]}"
            )

        self.step += 1
        self.root.after(
            500,
            self.animate
        )

root = tk.Tk()
App(root)
root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8,0)
MAX_DEPTH = 50
ANIMATION_DELAY = 450
class PuzzleIDS:

    def __init__(self):
        self.window = tk.Tk()
        self.window.title("8 Puzzle - IDS")
        self.current_board = self.generate_board()
        self.solution_path = []
        self.move_history = []
        self.animation_index = 0

        self.build_ui()
        self.update_board(self.current_board)
    def generate_children(self, state):
        empty_index = state.index(0)
        row, col = divmod(empty_index, 3)
        next_states = []
        directions = [
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1)
        ]
        for dr, dc in directions:
            new_row = row + dr
            new_col = col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                swap_index = new_row * 3 + new_col
                temp = list(state)
                temp[empty_index], temp[swap_index] = (
                    temp[swap_index],
                    temp[empty_index]
                )
                next_states.append(tuple(temp))
        return next_states

    def get_direction(self, old_state, new_state):
        old_pos = old_state.index(0)
        new_pos = new_state.index(0)
        diff = new_pos - old_pos
        if diff == -3:
            return "UP"
        if diff == 3:
            return "DOWN"
        if diff == -1:
            return "LEFT"
        return "RIGHT"

    def is_solvable(self, board):
        values = [x for x in board if x != 0]
        inversions = 0
        for i in range(len(values)):
            for j in range(i + 1, len(values)):
                if values[i] > values[j]:
                    inversions += 1
        return inversions % 2 == 0

    def generate_board(self):
        while True:
            board = list(GOAL_STATE)
            random.shuffle(board)
            board = tuple(board)
            if board != GOAL_STATE and self.is_solvable(board):
                return board

    def ids(self, start_state, limit):
        stack = [(start_state, [start_state])]
        expanded_nodes = 0
        while stack:
            current_state, path = stack.pop()
            expanded_nodes += 1
            if current_state == GOAL_STATE:
                return path, expanded_nodes
            current_depth = len(path) - 1
            if current_depth < limit:
                children = self.generate_children(current_state)
                for child in children:
                    if child not in path:
                        stack.append((child, path + [child]))
        return None, expanded_nodes

    def iterative_deepening_search(self, start_state):
        total_nodes = 0
        for depth in range(MAX_DEPTH):
            result, nodes = self.ids(
                start_state,
                depth
            )
            total_nodes += nodes
            if result:
                return result, total_nodes

        return None, total_nodes
    def build_ui(self):

        container = tk.Frame(self.window)
        container.pack(padx=12, pady=12)

        left_panel = tk.Frame(container)
        left_panel.pack(side="left", padx=10)

        self.board_frame = tk.Frame(left_panel)
        self.board_frame.pack()

        self.cells = []

        for i in range(9):

            label = tk.Label(
                self.board_frame,
                width=4,
                height=2,
                font=("Arial", 24, "bold"),
                relief="solid",
                borderwidth=2
            )

            label.grid(
                row=i // 3,
                column=i % 3,
                padx=2,
                pady=2
            )

            self.cells.append(label)

        self.status_label = tk.Label(
            left_panel,
            text="Ready",
            font=("Arial", 12)
        )

        self.status_label.pack(pady=10)

        button_frame = tk.Frame(left_panel)
        button_frame.pack()

        tk.Button(
            button_frame,
            text="Shuffle",
            width=12,
            command=self.shuffle_board
        ).pack(side="left", padx=5)

        tk.Button(
            button_frame,
            text="Solve IDS",
            width=12,
            command=self.solve_puzzle
        ).pack(side="left", padx=5)

        right_panel = tk.Frame(container)
        right_panel.pack(side="right", padx=10)

        tk.Label(
            right_panel,
            text="Move List",
            font=("Arial", 14, "bold")
        ).pack()

        self.move_box = tk.Text(
            right_panel,
            width=22,
            height=20,
            font=("Consolas", 11)
        )

        self.move_box.pack()
        self.info_label = tk.Label(
            self.window,
            text="",
            font=("Arial", 11)
        )

        self.info_label.pack(pady=5)

    def update_board(self, board):
        for i, value in enumerate(board):
            self.cells[i]["text"] = "" if value == 0 else str(value)

    def shuffle_board(self):
        self.current_board = self.generate_board()
        self.solution_path = []
        self.move_history = []
        self.animation_index = 0
        self.move_box.delete("1.0", tk.END)
        self.status_label["text"] = "Ready"
        self.info_label["text"] = ""
        self.update_board(self.current_board)

    def solve_puzzle(self):
        start_time = time.perf_counter()
        path, nodes = self.iterative_deepening_search(
            self.current_board
        )
        elapsed_ms = (
            time.perf_counter() - start_time
        ) * 100
        if not path:
            return

        self.solution_path = path
        self.move_history = []

        for i in range(len(path) - 1):
            move = self.get_direction(
                path[i],
                path[i + 1]
            )

            self.move_history.append(move)
        self.animation_index = 0

        self.info_label["text"] = (
            f"Expanded Nodes: {nodes}    "
            f"Steps: {len(path) - 1}    "
            f"Time: {elapsed_ms:.2f} ms"
        )
        self.move_box.delete("1.0", tk.END)
        for step, move in enumerate(self.move_history, start=1):

            self.move_box.insert(
                tk.END,
                f"{step}. {move}\n"
            )
        self.animate_solution()

    def animate_solution(self):
        if self.animation_index >= len(self.solution_path):
            return
        board = self.solution_path[self.animation_index]
        self.update_board(board)
        if self.animation_index == 0:
            self.status_label["text"] = "Initial State"
        else:
            current_move = self.move_history[
                self.animation_index - 1
            ]
            self.status_label["text"] = (
                f"Step {self.animation_index}: "
                f"{current_move}"
            )

        self.animation_index += 1
        self.window.after(
            ANIMATION_DELAY,
            self.animate_solution
        )

    def run(self):

        self.window.mainloop()
if __name__ == "__main__":

    app = PuzzleIDS()
    app.run()

In [ ]:
import tkinter as tk
import heapq
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

POSITION_COST = {
    0: 4, 1: 2, 2: 4,
    3: 2, 4: 1, 5: 2,
    6: 4, 7: 2, 8: 4
}

def step_cost(board):
    return POSITION_COST[board.index(0)]

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def ucs(start):
    counter = 0
    heap = [(0, counter, start, [start], [0])]
    reached = {start: 0}
    nodes = 0

    while heap:
        g, _, board, path, costs = heapq.heappop(heap)
        nodes += 1
        if board == GOAL:
            return path, costs, nodes
        if g > reached.get(board, float('inf')):
            continue
        for child in get_neighbors(board):
            child_g = g + step_cost(child)
            if child_g < reached.get(child, float('inf')):
                reached[child] = child_g
                counter += 1
                heapq.heappush(heap, (
                    child_g, counter, child,
                    path + [child], costs + [child_g]
                ))
    return None, [], nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_step_cost = tk.Label(
            left, text="Step cost: —",
            bg="white", fg="#7c5cbf", font=("Arial", 10)
        )
        self.lb_step_cost.pack()

        self.lb_total_cost = tk.Label(
            left, text="Total path cost: 0", bg="white", fg="blue", font=("Arial", 10)
        )
        self.lb_total_cost.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(right, text="Solution", font=("Arial", 14, "bold"), bg="white").pack()

        self.solution_box = tk.Text(right, width=22, height=22, font=("Consolas", 10))
        self.solution_box.pack()

        tk.Label(right, text="Position cost map:", font=("Arial", 9), bg="white", fg="gray").pack(pady=(6,0))
        legend = tk.Canvas(right, width=120, height=120, bg="white", highlightthickness=0)
        legend.pack()
        self._draw_legend(legend)

        self.draw()

    def _draw_legend(self, canvas):
        colors = {1: "#e8f5e9", 2: "#fff9c4", 4: "#fce4ec"}
        s = 36
        for i in range(9):
            r, c = divmod(i, 3)
            x1, y1 = c * s + 2, r * s + 2
            x2, y2 = x1 + s - 2, y1 + s - 2
            cost = POSITION_COST[i]
            canvas.create_rectangle(x1, y1, x2, y2, fill=colors[cost], outline="#ccc")
            canvas.create_text(
                (x1 + x2) // 2, (y1 + y2) // 2,
                text=str(cost), font=("Arial", 11, "bold"),
                fill="#333"
            )

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        blank_idx = board.index(0)
        cost_val = POSITION_COST[blank_idx]

        pos_colors = {1: "#e8f5e9", 2: "#fff9c4", 4: "#fce4ec"}

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                fill = pos_colors[POSITION_COST[i]]
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="#aaa",
                    width=2, dash=(4, 3)
                )
                # Không hiển thị gì trong ô trống
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )
        self.lb_step_cost.config(text=f"Step cost: {cost_val}")

        if self.costs and self.solution:
            self.lb_total_cost.config(text=f"Total path cost: {self.costs[self.step]}")
        else:
            self.lb_total_cost.config(text="Total path cost: 0")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_step_cost.config(text="Step cost: —")
        self.lb_total_cost.config(text="Total path cost: 0")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, costs, nodes = ucs(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            return

        self.solution = path
        self.costs = costs
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")

        self.solution_box.delete("1.0", tk.END)
        for i, m in enumerate(self.moves, 1):
            blank_after = path[i].index(0)
            sc = POSITION_COST[blank_after]
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} g={costs[i]}  (+{sc})\n"
            )

        self.playing = True
        self.animate()

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        if self.step > 0:
            line = f"{self.step}.0"
            self.solution_box.tag_add("highlight", line, f"{self.step}.end")
            self.solution_box.tag_config("highlight", background="#cce5ff")
            self.solution_box.see(line)

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self.solution_box.tag_remove("highlight", "1.0", tk.END)
            self.solution_box.tag_add("highlight", f"{self.step}.0", f"{self.step}.end")
            self.solution_box.tag_config("highlight", background="#cce5ff")
            self.solution_box.see(f"{self.step}.0")

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import heapq
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
ANIM = 300

GOAL_POS = {
    val: divmod(i, 3)
    for i, val in enumerate(GOAL)
}
def manhattan(board):
    total = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        total += abs(r - gr) + abs(c - gc)
    return total

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def greedy(start):
    counter = 0
    h0 = manhattan(start)
    heap = [(h0,counter, start,[start])]
    reached = {start}
    nodes = 0
    while heap:
        h, _, board, path = heapq.heappop(heap)
        nodes += 1
        if board == GOAL:
            return path, nodes
        for child in get_neighbors(board):
            if child not in reached:
                reached.add(child)
                counter += 1
                heapq.heappush(
                    heap,
                    ( manhattan(child),counter,child,path + [child]
                    )
                )
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if t != GOAL and is_solvable(t):
            return t

class App:

    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False
        main = tk.Frame(root,bg="white")
        main.pack(padx=10,pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left",padx=10)
        self.canvas = tk.Canvas(left, width=3 * TILE,height=3 * TILE, bg="white",highlightthickness=0)
        self.canvas.pack()
        self.step_label = tk.Label(left,text="Steps:",font=("Arial", 12),bg="white")
        self.step_label.pack(pady=6)
        self.lb_h = tk.Label(left,text="h(n): 0",bg="white",fg="blue",font=("Arial", 11))
        self.lb_h.pack()
        info = tk.Frame( left,bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info,text="Nodes: 0",bg="white")
        self.lb_nodes.grid( row=0,column=0,padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid( row=0,column=1,padx=6)
        self.lb_time = tk.Label( info, text="Time: 0 ms",bg="white")
        self.lb_time.grid(row=0,column=2,padx=6)
        btns = tk.Frame( left, bg="white")
        btns.pack(pady=6)

        tk.Button( btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0,column=0,padx=4)
        tk.Button(btns,text="Reset", width=10,command=self.reset).grid( row=0,column=1,padx=4)
        tk.Button(btns,text="Solve",width=10,command=self.solve).grid(  row=0,  column=2,  padx=4)
        tk.Button( btns, text="Step",width=34, command=self.next_step).grid(row=1,column=0,columnspan=3,pady=4 )

        right = tk.Frame( main,bg="white")
        right.pack(side="right",padx=10)

        tk.Label(right,text="Solution",font=("Arial", 14, "bold"),bg="white").pack()

        self.solution_box = tk.Text(right,width=22,height=20,font=("Consolas", 10))
        self.solution_box.pack()
        self.draw()

    def get_direction(self,old_state,new_state):
        diff = (new_state.index(0)-old_state.index(0)
        )
        return {
            -3: "UP",
             3: "DOWN",
            -1: "LEFT",
             1: "RIGHT"
        }.get(diff, "?")
    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = c * TILE
            y1 = r * TILE
            x2 = x1 + TILE
            y2 = y1 + TILE
            if val == 0:
                self.canvas.create_rectangle(
                    x1,
                    y1,
                    x2,
                    y2,
                    fill="#f2f2f2",
                    outline="black",
                    width=2
                )
            else:
                wrong = ( val != GOAL[i])
                self.canvas.create_rectangle( x1, y1, x2,y2, fill="white", outline="black",width=2)
                self.canvas.create_text(
                    (x1 + x2) // 2,
                    (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28, "bold"),
                    fill="red" if wrong else "black"
                )

        self.lb_h.config(
            text=f"h(n): {manhattan(board)}"
        )

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete( "1.0",tk.END)
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.draw()

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = greedy(self.board)
        ms = ( time.perf_counter() - t0 ) * 1000
        if path is None:
            return
        self.solution = path
        self.moves = []
        for i in range(len(path) - 1):
            self.moves.append(
                self.get_direction(
                    path[i],
                    path[i + 1]
                )
            )

        self.step = 0
        self.lb_nodes.config( text=f"Nodes: {nodes}")
        self.lb_steps.config( text=f"Steps: {len(path)-1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.solution_box.delete( "1.0", tk.END)
        self.playing = True
        self.animate()

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config( text="START")
        else:
            self.step_label.config(
                text=f"Step {self.step}: "
                f"{self.moves[self.step-1]}"
            )

        self.solution_box.tag_remove(
            "highlight",
            "1.0",
            tk.END
        )
        self.solution_box.insert(
            tk.END,
            f"{self.step}. {self.moves[self.step-1]}   h(n)={manhattan(self.solution[self.step])}\n"
        )
        if self.step > 0:
            line = f"{self.step}.0"
            self.solution_box.tag_remove("highlight","1.0",tk.END)
            self.solution_box.tag_add(
                "highlight",
                line,
                f"{self.step}.end"
            )
            
            self.solution_box.tag_config(
                "highlight",
                background="#cce5ff"
            )
            self.solution_box.see(line)
        if self.step < len(self.solution) - 1:
            self.root.after(ANIM,self.animate)
            self.step += 1
            
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(
                text=f"Step {self.step}: "
                f"{self.moves[self.step-1]}"
            )
            

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import heapq
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

POSITION_COST = {
    0: 4, 1: 2, 2: 4,
    3: 2, 4: 1, 5: 2,
    6: 4, 7: 2, 8: 4
}
GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}
def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def step_cost(board):
    return POSITION_COST[board.index(0)]

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def astar(start):
    counter = 0
    h0 = manhattan(start)
    heap = [(h0, counter, 0, start, [start], [0])]
    reached = {start: 0}
    nodes = 0
    while heap:
        f, _, g, board, path, costs = heapq.heappop(heap)
        nodes += 1
        if board == GOAL:
            return path, costs, nodes
        if g > reached.get(board, float('inf')):
            continue
        for child in get_neighbors(board):
            child_g = g + step_cost(child)
            if child_g < reached.get(child, float('inf')):
                reached[child] = child_g
                counter += 1
                child_f = child_g + manhattan(child)
                heapq.heappush(heap, (
                    child_f, counter, child_g, child,
                    path + [child], costs + [child_g]
                ))
    return None, [], nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — A*")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_step_cost = tk.Label(
            left, text="Step cost: —",
            bg="white", fg="#7c5cbf", font=("Arial", 10)
        )
        self.lb_step_cost.pack()

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        self.lb_total_cost = tk.Label(
            left, text="g(n) = 0  |  f(n) = —", bg="white", fg="blue", font=("Arial", 10)
        )
        self.lb_total_cost.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(right, text="Solution (A*)", font=("Arial", 14, "bold"), bg="white").pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        blank_idx = board.index(0)
        cost_val = POSITION_COST[blank_idx]
        h_val = manhattan(board)

        pos_colors = {1: "#e8f5e9", 2: "#fff9c4", 4: "#fce4ec"}

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                fill = pos_colors[POSITION_COST[i]]
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_step_cost.config(text=f"Step cost: {cost_val}")
        self.lb_heuristic.config(text=f"h(n) = {h_val}")

        if self.costs and self.solution:
            g = self.costs[self.step]
            f = g + h_val
            self.lb_total_cost.config(text=f"g(n) = {g}  |  f(n) = {f}")
        else:
            self.lb_total_cost.config(text=f"g(n) = 0  |  f(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_step_cost.config(text="Step cost: —")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_total_cost.config(text="g(n) = 0  |  f(n) = —")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, costs, nodes = astar(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            return

        self.solution = path
        self.costs = costs
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")

        self.solution_box.delete("1.0", tk.END)

        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            board_after = self.solution[i]
            sc = POSITION_COST[board_after.index(0)]
            h = manhattan(board_after)
            g = self.costs[i]
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} g={g}(+{sc}) h={h} f={g+h}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        last_line = f"{self.step}.0"
        self.solution_box.tag_add("highlight", last_line, f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
ANIM = 300

GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}
def h_misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])


def manhattan(board):
    total = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        total += abs(r - gr) + abs(c - gc)
    return total

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result


def get_direction(old_state, new_state):
    diff = new_state.index(0) - old_state.index(0)
    return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(1 for i in range(len(arr)) for j in range(i + 1, len(arr)) if arr[i] > arr[j])
    return inv % 2 == 0


def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if t != GOAL and is_solvable(t):
            return t

def ida_star(start):
    threshold = h_misplaced(start)
    path = [start]
    total_nodes = [0]

    def search(g):
        nonlocal threshold
        current = path[-1]
        f = g + h_misplaced(current)
        total_nodes[0] += 1
        if f > threshold:
            return f        
        if current == GOAL:
            return "FOUND"

        minimum = float('inf')
        for child in get_neighbors(current):
            if child not in path:
                path.append(child)
                result = search(g + 1)
                if result == "FOUND":
                    return "FOUND"
                if result < minimum:
                    minimum = result
                path.pop()

        return minimum

    while True:
        result = search(0)
        if result == "FOUND":
            return list(path), total_nodes[0]
        if result == float('inf'):
            return None, total_nodes[0]
        threshold = result

class App:

    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle - IDA*")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False
        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)
        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left, width=3 * TILE, height=3 * TILE,
            bg="white", highlightthickness=0
        )
        self.canvas.pack()
        self.step_label = tk.Label(left, text="Steps:", font=("Arial", 12), bg="white")
        self.step_label.pack(pady=6)
        self.lb_h = tk.Label(left, text="h(n): 0", bg="white", fg="blue", font=("Arial", 11))
        self.lb_h.pack()
        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)
        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)

        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step", width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)
        tk.Label(right, text="Solution", font=("Arial", 14, "bold"), bg="white").pack()
        self.solution_box = tk.Text(right, width=22, height=20, font=("Consolas", 10))
        self.solution_box.pack()
        self.draw()

    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1, y1 = c * TILE, r * TILE
            x2, y2 = x1 + TILE, y1 + TILE
            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#f2f2f2", outline="black", width=2
                )
            else:
                wrong = (val != GOAL[i])
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="white", outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28, "bold"),
                    fill="red" if wrong else "black"
                )
        self.lb_h.config(text=f"h(n): {h_misplaced(board)}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.draw()

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = ida_star(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            self.step_label.config(text="No solution!")
            return

        self.solution = path
        self.moves = [get_direction(path[i], path[i + 1]) for i in range(len(path) - 1)]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.solution_box.delete("1.0", tk.END)
        self.playing = True
        self.animate()

    def animate(self):
        if not self.playing:
            return

        self.draw()

        if self.step == 0:
            self.step_label.config(text="START")
        else:
            g = self.step
            h = h_misplaced(self.solution[self.step])
            self.step_label.config(
                text=f"Step {self.step}: {self.moves[self.step - 1]}   "
                     f"g={g}  h={h}  f={g + h}"
            )

        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        if self.step < len(self.moves):
            h_val = h_misplaced(self.solution[self.step])
            g_val = self.step
            self.solution_box.insert(
                tk.END,
                f"{self.step}. {self.moves[self.step - 1] if self.step > 0 else '---'}"
                f"   h={h_val}  g={g_val}  f={g_val + h_val}\n"
            )

        if self.step > 0:
            line = f"{self.step}.0"
            self.solution_box.tag_add("highlight", line, f"{self.step}.end")
            self.solution_box.tag_config("highlight", background="#cce5ff")
            self.solution_box.see(line)

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            g = self.step
            h = h_misplaced(self.solution[self.step])
            self.step_label.config(
                text=f"Step {self.step}: {self.moves[self.step - 1]}   "
                     f"g={g}  h={h}  f={g + h}"
            )


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

POSITION_COST = {
    0: 4, 1: 2, 2: 4,
    3: 2, 4: 1, 5: 2,
    6: 4, 7: 2, 8: 4
}
GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}
def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def step_cost(board):
    return POSITION_COST[board.index(0)]

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def hill_climbing(start):
    current = start
    path = [current]
    costs = [0]
    total_cost = 0
    nodes = 1
    while current != GOAL:
        current_h = manhattan(current)
        found_better = False
        for child in get_neighbors(current):
            nodes += 1
            if manhattan(child) < current_h:
                total_cost += step_cost(child)
                current = child
                path.append(current)
                costs.append(total_cost)
                found_better = True
                break

        if not found_better:
            break

    return path, costs, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Hill Climbing")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_step_cost = tk.Label(
            left, text="Step cost: —",
            bg="white", fg="#7c5cbf", font=("Arial", 10)
        )
        self.lb_step_cost.pack()

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        self.lb_total_cost = tk.Label(
            left, text="g(n) = 0  |  f(n) = —", bg="white", fg="blue", font=("Arial", 10)
        )
        self.lb_total_cost.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(right, text="Solution (Hill Climbing)", font=("Arial", 14, "bold"), bg="white").pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        blank_idx = board.index(0)
        cost_val = POSITION_COST[blank_idx]
        h_val = manhattan(board)

        pos_colors = {1: "#e8f5e9", 2: "#fff9c4", 4: "#fce4ec"}

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                fill = pos_colors[POSITION_COST[i]]
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_step_cost.config(text=f"Step cost: {cost_val}")
        self.lb_heuristic.config(text=f"h(n) = {h_val}")

        if self.costs and self.solution:
            g = self.costs[self.step]
            f = g + h_val
            self.lb_total_cost.config(text=f"g(n) = {g}  |  f(n) = {f}")
        else:
            self.lb_total_cost.config(text=f"g(n) = 0  |  f(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_step_cost.config(text="Step cost: —")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_total_cost.config(text="g(n) = 0  |  f(n) = —")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, costs, nodes = hill_climbing(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            return

        self.solution = path
        self.costs = costs
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")

        self.solution_box.delete("1.0", tk.END)

        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            board_after = self.solution[i]
            sc = POSITION_COST[board_after.index(0)]
            h = manhattan(board_after)
            g = self.costs[i]
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} g={g}(+{sc}) h={h} f={g+h}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        last_line = f"{self.step}.0"
        self.solution_box.tag_add("highlight", last_line, f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

POSITION_COST = {
    0: 4, 1: 2, 2: 4,
    3: 2, 4: 1, 5: 2,
    6: 4, 7: 2, 8: 4
}
GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}
def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def step_cost(board):
    return POSITION_COST[board.index(0)]

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def hill_climbing(start):
    current = start
    path = [current]
    costs = [0]
    total_cost = 0
    nodes = 1
    while current != GOAL:

        current_h = manhattan(current)
        neighbors = get_neighbors(current)
        nodes += len(neighbors)
        best_neighbor = min(
            neighbors,
            key=lambda state: manhattan(state)
        )

        best_h = manhattan(best_neighbor)
        if best_h >= current_h:
            break
        total_cost += step_cost(best_neighbor)
        current = best_neighbor
        path.append(current)
        costs.append(total_cost)

    return path, costs, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Steepest-Ascent Hill Climbing")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_step_cost = tk.Label(
            left, text="Step cost: —",
            bg="white", fg="#7c5cbf", font=("Arial", 10)
        )
        self.lb_step_cost.pack()

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        self.lb_total_cost = tk.Label(
            left, text="g(n) = 0  |  f(n) = —", bg="white", fg="blue", font=("Arial", 10)
        )
        self.lb_total_cost.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(right, text="Solution (Steepest-Ascent Hill Climbing)", font=("Arial", 14, "bold"), bg="white").pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        blank_idx = board.index(0)
        cost_val = POSITION_COST[blank_idx]
        h_val = manhattan(board)

        pos_colors = {1: "#e8f5e9", 2: "#fff9c4", 4: "#fce4ec"}

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                fill = pos_colors[POSITION_COST[i]]
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_step_cost.config(text=f"Step cost: {cost_val}")
        self.lb_heuristic.config(text=f"h(n) = {h_val}")

        if self.costs and self.solution:
            g = self.costs[self.step]
            f = g + h_val
            self.lb_total_cost.config(text=f"g(n) = {g}  |  f(n) = {f}")
        else:
            self.lb_total_cost.config(text=f"g(n) = 0  |  f(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_step_cost.config(text="Step cost: —")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_total_cost.config(text="g(n) = 0  |  f(n) = —")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, costs, nodes = hill_climbing(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            return

        self.solution = path
        self.costs = costs
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")

        self.solution_box.delete("1.0", tk.END)

        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            board_after = self.solution[i]
            sc = POSITION_COST[board_after.index(0)]
            h = manhattan(board_after)
            g = self.costs[i]
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} cost={g}(+{sc}) h={h}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        last_line = f"{self.step}.0"
        self.solution_box.tag_add("highlight", last_line, f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}

def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def random_hill_climbing(start, max_steps=2000):
    current = start
    path = [current]

    for _ in range(max_steps):
        if current == GOAL:
            return path
        current_h = manhattan(current)
        better = [n for n in get_neighbors(current) if manhattan(n) < current_h]
        if not better:
            break
        current = random.choice(better)
        path.append(current)

    return path


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Random Hill Climbing")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=0, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Random Hill Climbing)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        h_val = manhattan(board)

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_heuristic.config(text=f"h(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_heuristic.config(text="h(n) = —")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path = random_hill_climbing(self.board)
        ms = (time.perf_counter() - t0) * 1000

        self.solution = path
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")

        solved = path[-1] == GOAL
        self.step_label.config(text="✓ SOLVED" if solved else "✗ Stuck")

        self.solution_box.delete("1.0", tk.END)
        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            h = manhattan(self.solution[i])
            self.solution_box.insert(tk.END, f"{i:2}. {m:<5} h(n)={h}\n")
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        self.solution_box.tag_add("highlight", f"{self.step}.0", f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            solved = self.solution[-1] == GOAL
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck (local optimum)")

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}

def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def random_hill_climbing(start, max_restarts=50, max_steps=1000):
    best_path = None
    total_nodes = 0
    restarts = 0
    current = start
    path = [current]

    for _ in range(max_steps):
        if current == GOAL:
            break

        current_h = manhattan(current)
        neighbors = get_neighbors(current)
        total_nodes += len(neighbors)

        better = [n for n in neighbors if manhattan(n) < current_h]

        if better:
            chosen = random.choice(better)
            current = chosen
            path.append(current)
        else:
            if restarts >= max_restarts:
                break
            restarts += 1

            if best_path is None or manhattan(current) < manhattan(best_path[-1]):
                best_path = path[:]
            current = random_board()
            path = [current]

    if current == GOAL:
        return path, total_nodes, restarts
    else:
        if best_path and manhattan(best_path[-1]) < manhattan(path[-1]):
            return best_path, total_nodes, restarts
        return path, total_nodes, restarts


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Random Hill Climbing")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        self.lb_restarts = tk.Label(left, text="Restarts: 0", bg="white", fg="#006400", font=("Arial", 10))
        self.lb_restarts.pack()

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(right, text="Solution (Random Hill Climbing)", font=("Arial", 14, "bold"), bg="white").pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        h_val = manhattan(board)

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_heuristic.config(text=f"h(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_restarts.config(text="Restarts: 0")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes, restarts = random_hill_climbing(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            return

        self.solution = path
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_restarts.config(text=f"Restarts: {restarts}")

        solved = path[-1] == GOAL
        status = "✓ SOLVED" if solved else "✗ Stuck"
        self.step_label.config(text=status)

        self.solution_box.delete("1.0", tk.END)

        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            board_after = self.solution[i]
            h = manhattan(board_after)
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} h(n)={h}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        last_line = f"{self.step}.0"
        self.solution_box.tag_add("highlight", last_line, f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            solved = self.solution[-1] == GOAL
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck (local optimum)")

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300
BEAM_WIDTH = 4  

GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}

def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def local_beam_search(start, k=BEAM_WIDTH, max_steps=2000):
    beams = [(start, [start])]
    for _ in range(k - 1):
        neighbors = get_neighbors(start)
        s = random.choice(neighbors)
        beams.append((s, [s]))
    visited = set(b for b, _ in beams)
    total_nodes = 0
    for _ in range(max_steps):
        for board, path in beams:
            if board == GOAL:
                return path, total_nodes, k
        all_neighbors = []
        for board, path in beams:
            for nb in get_neighbors(board):
                total_nodes += 1
                if nb not in visited:
                    all_neighbors.append((nb, path + [nb]))
                    visited.add(nb)

        if not all_neighbors:
            break
        all_neighbors.sort(key=lambda x: manhattan(x[0]))
        beams = all_neighbors[:k]
    best_board, best_path = min(beams, key=lambda x: manhattan(x[0]))
    return best_path, total_nodes, k


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Local Beam Search")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        info2 = tk.Frame(left, bg="white")
        info2.pack(pady=2)
        self.lb_beam = tk.Label(
            info2, text=f"Beam: {BEAM_WIDTH}", bg="white", fg="#006400", font=("Arial", 10)
        )
        self.lb_beam.grid(row=0, column=0, padx=6)
        self.lb_visited = tk.Label(
            info2, text="Visited: 0", bg="white", fg="#00008b", font=("Arial", 10)
        )
        self.lb_visited.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Local Beam Search)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        h_val = manhattan(board)

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_heuristic.config(text=f"h(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_beam.config(text=f"Beam: {BEAM_WIDTH}")
        self.lb_visited.config(text="Visited: 0")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes, beam_k = local_beam_search(self.board)
        ms = (time.perf_counter() - t0) * 1000

        self.solution = path
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_beam.config(text=f"Beam: {beam_k}")
        self.lb_visited.config(text=f"Visited: {nodes}")

        solved = path[-1] == GOAL
        self.step_label.config(text="✓ SOLVED" if solved else "✗ Stuck")

        self.solution_box.delete("1.0", tk.END)
        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            h = manhattan(self.solution[i])
            self.solution_box.insert(tk.END, f"{i:2}. {m:<5} h(n)={h}\n")
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        self.solution_box.tag_add("highlight", f"{self.step}.0", f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            solved = self.solution[-1] == GOAL
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck (local optimum)")

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import math
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

# Heuristic: số ô sai vị trí (không tính ô trống)
def misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def simulated_annealing(start, T=100.0, alpha=0.995, T_min=0.01, max_steps=50000):
    current = start
    path = [current]
    T_cur = T
    total_nodes = 0

    while T_cur > T_min and len(path) < max_steps:
        if current == GOAL:
            break

        neighbors = get_neighbors(current)
        total_nodes += len(neighbors)
        next_state = random.choice(neighbors)

        delta = misplaced(next_state) - misplaced(current)

        if delta < 0:
            current = next_state
            path.append(current)
        else:
            prob = math.exp(-delta / T_cur)
            if random.random() < prob:
                current = next_state
                path.append(current)

        T_cur *= alpha

    return path, total_nodes, T_cur


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Simulated Annealing")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        info2 = tk.Frame(left, bg="white")
        info2.pack(pady=2)
        self.lb_temp = tk.Label(
            info2, text="T₀: 100.0", bg="white", fg="#006400", font=("Arial", 10)
        )
        self.lb_temp.grid(row=0, column=0, padx=6)
        self.lb_visited = tk.Label(
            info2, text="Visited: 0", bg="white", fg="#00008b", font=("Arial", 10)
        )
        self.lb_visited.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Simulated Annealing)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        h_val = misplaced(board)

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_heuristic.config(text=f"h(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_temp.config(text="T₀: 100.0")
        self.lb_visited.config(text="Visited: 0")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes, T_final = simulated_annealing(self.board)
        ms = (time.perf_counter() - t0) * 1000

        self.solution = path
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_temp.config(text=f"T_end: {T_final:.4f}")
        self.lb_visited.config(text=f"Visited: {nodes}")

        solved = path[-1] == GOAL
        self.step_label.config(text="✓ SOLVED" if solved else "✗ Stuck")

        self.solution_box.delete("1.0", tk.END)
        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            h = misplaced(self.solution[i])
            self.solution_box.insert(tk.END, f"{i:2}. {m:<5} h(n)={h}\n")
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        self.solution_box.tag_add("highlight", f"{self.step}.0", f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            solved = self.solution[-1] == GOAL
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck (local optimum)")

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP  = 5
ANIM = 450

def misplaced(b):
    return sum(1 for i,v in enumerate(b) if v!=0 and v!=GOAL[i])

def apply_action(b, a):
    idx = b.index(0)
    r,c = divmod(idx,3)
    dr,dc = a
    nr,nc = r+dr, c+dc
    if 0<=nr<3 and 0<=nc<3:
        ni = nr*3+nc
        t = list(b); t[idx],t[ni]=t[ni],t[idx]
        return tuple(t)
    return None

ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ANAMES  = {(-1,0):"UP",(1,0):"DOWN",(0,-1):"LEFT",(0,1):"RIGHT"}
def and_or_graph_search(initial):
    counter = [0]

    def OR_search(s, path, g, limit):
        if s == GOAL:
            return [], counter[0]
        f = g + misplaced(s)
        if f > limit:
            return None, counter[0]       
        if s in path:
            return None, counter[0]          
        counter[0] += 1
        children = []
        for a in ACTIONS:
            nb = apply_action(s, a)
            if nb is not None:
                children.append((misplaced(nb), a, nb))
        children.sort(key=lambda x: x[0])

        new_path = path | {s}
        for _, a, nb in children:
            plan, _ = AND_search({nb}, new_path, g+1, limit, a, s)
            if plan is not None:
                return plan, counter[0]
        return None, counter[0]         
    
    def AND_search(outcomes, path, g, limit, action, parent):
        for nb in outcomes:
            sub, _ = OR_search(nb, path, g, limit)
            if sub is None:
                return None, counter[0]  
            return {"action": action, "next": nb, "sub": sub}, counter[0]
        return None, counter[0]
    
    limit = misplaced(initial)
    plan  = None
    nodes = 0
    while limit <= 80:
        counter[0] = 0
        plan, nodes = OR_search(initial, frozenset(), 0, limit)
        nodes = counter[0]
        if plan is not None:
            break
        limit += 1

    path_states = [initial]
    path_acts   = []
    p = plan
    while isinstance(p, dict):
        path_states.append(p["next"])
        path_acts.append(ANAMES[p["action"]])
        p = p["sub"]

    depth = len(path_states)-1
    return plan, nodes, path_states, path_acts, depth

def is_solvable(b):
    arr = [x for x in b if x!=0]
    inv = sum(1 for i in range(len(arr)) for j in range(i+1,len(arr)) if arr[i]>arr[j])
    return inv%2==0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t!=GOAL:
            return t
class App:
    def __init__(self, root):
        self.root   = root
        self.root.title("8 Puzzle — AND-OR Graph Search")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.path  = []
        self.moves = []
        self.step  = 0
        self.playing = False

        main = tk.Frame(root, bg="white"); main.pack(padx=10, pady=10)
        left = tk.Frame(main, bg="white"); left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(left, width=3*TILE+4*GAP, height=3*TILE+4*GAP,
                                bg="white", highlightthickness=0)
        self.canvas.pack()

        self.lbl_step = tk.Label(left, text="", font=("Arial",12), bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h = tk.Label(left, text="h(n) = —", bg="white",
                              fg="#c05000", font=("Arial",10))
        self.lbl_h.pack()

        nf = tk.Frame(left, bg="white", bd=1, relief="groove")
        nf.pack(pady=4, fill="x", padx=4)
        tk.Label(nf, text="Node type:", bg="white",
                 font=("Arial",9,"bold")).pack(side="left", padx=4)
        self.lbl_ntype = tk.Label(nf, text="OR node", bg="white",
                                  fg="#0055cc", font=("Consolas",10))
        self.lbl_ntype.pack(side="left", padx=4)

        r1 = tk.Frame(left, bg="white"); r1.pack(pady=2)
        self.lbl_nodes = tk.Label(r1, text="Nodes: —", bg="white"); self.lbl_nodes.grid(row=0,column=0,padx=5)
        self.lbl_steps = tk.Label(r1, text="Steps: —", bg="white"); self.lbl_steps.grid(row=0,column=1,padx=5)
        self.lbl_time  = tk.Label(r1, text="Time: — ms", bg="white"); self.lbl_time.grid(row=0,column=2,padx=5)

        r2 = tk.Frame(left, bg="white"); r2.pack(pady=2)
        self.lbl_depth = tk.Label(r2, text="Depth: —", bg="white",
                                  fg="#006400", font=("Arial",10))
        self.lbl_depth.grid(row=0,column=0,padx=5)
        self.lbl_path = tk.Label(r2, text="P=[S]", bg="white",
                                 fg="#00008b", font=("Arial",10))
        self.lbl_path.grid(row=0,column=1,padx=5)

        btns = tk.Frame(left, bg="white"); btns.pack(pady=6)
        tk.Button(btns,text="Shuffle",width=10,command=self.shuffle).grid(row=0,column=0,padx=4)
        tk.Button(btns,text="Reset",  width=10,command=self.reset  ).grid(row=0,column=1,padx=4)
        tk.Button(btns,text="Solve",  width=10,command=self.solve  ).grid(row=0,column=2,padx=4)
        tk.Button(btns,text="Step ▶", width=34,command=self.next_step).grid(
            row=1,column=0,columnspan=3,pady=4)
        right = tk.Frame(main, bg="white"); right.pack(side="right", padx=10)
        tk.Label(right, text="Solution (AND-OR Graph Search)",
                 font=("Arial",13,"bold"), bg="white").pack()
        self.sol = tk.Text(right, width=30, height=25, font=("Consolas",9))
        self.sol.pack()
        self.draw()


    def cur_board(self):
        if self.path and self.step < len(self.path):
            return self.path[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        b = self.cur_board()
        self.lbl_h.config(text=f"h(n) = {misplaced(b)}")
        for i,val in enumerate(b):
            r,c = divmod(i,3)
            x1=GAP+c*(TILE+GAP); y1=GAP+r*(TILE+GAP)
            x2=x1+TILE;          y2=y1+TILE
            if val==0:
                self.canvas.create_rectangle(x1,y1,x2,y2,fill="#e8f5e9",
                    outline="#aaa",width=2,dash=(4,3))
            else:
                w = val!=GOAL[i]
                self.canvas.create_rectangle(x1,y1,x2,y2,
                    fill="#ffe0e0" if w else "#f5f5f5",
                    outline="#cc0000" if w else "#999",width=2)
                self.canvas.create_text((x1+x2)//2,(y1+y2)//2,
                    text=str(val),font=("Arial",28),
                    fill="red" if w else "black")

    def _reset(self):
        self.path=[]; self.moves=[]; self.step=0; self.playing=False
        self.lbl_nodes.config(text="Nodes: —")
        self.lbl_steps.config(text="Steps: —")
        self.lbl_time .config(text="Time: — ms")
        self.lbl_depth.config(text="Depth: —")
        self.lbl_path .config(text="P=[S]")
        self.lbl_step .config(text="")
        self.lbl_h    .config(text="h(n) = —")
        self.lbl_ntype.config(text="OR node", fg="#0055cc")
        self.sol.delete("1.0",tk.END)

    def shuffle(self):
        self.board=random_board(); self._reset(); self.draw()

    def reset(self):
        self.board=GOAL; self._reset(); self.draw()

    def _path_str(self):
        L = "SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        n = min(self.step+1, len(L))
        return "P=[" + ",".join(L[i] for i in range(n)) + "]"
    def solve(self):
        if self.playing: return
        self._reset()
        self.draw()

        t0 = time.perf_counter()
        plan, nodes, path_states, path_acts, depth = and_or_graph_search(self.board)
        ms = (time.perf_counter()-t0)*1000

        self.path  = path_states
        self.moves = path_acts
        self.step  = 0

        solved = bool(path_states) and path_states[-1]==GOAL
        self.lbl_nodes.config(text=f"Nodes: {nodes}")
        self.lbl_steps.config(text=f"Steps: {depth}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_depth.config(text=f"Depth: {depth}")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed")

        if path_states:
            self.playing=True
            self.animate()
    def _update_sol(self):
        if not self.path or self.step==0: return
        self.sol.delete("1.0",tk.END)
        L = "SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        for i in range(1, self.step+1):
            m  = self.moves[i-1] if i-1<len(self.moves) else "?"
            h  = misplaced(self.path[i])
            nm = L[i] if i<len(L) else f"N{i}"
            prev = L[i-1] if i-1<len(L) else f"N{i-1}"
            self.sol.insert(tk.END,
                f"{i:2}. [OR ] {prev} →{m:<5}\n")
            self.sol.insert(tk.END,
                f"    [AND] →{nm}  h({nm})={h}\n")

        self.sol.tag_remove("hl","1.0",tk.END)
        line = self.step*2-1
        self.sol.tag_add("hl",f"{line}.0",f"{line+2}.end")
        self.sol.tag_config("hl",background="#cce5ff")
        self.sol.see(tk.END)
        self.lbl_path.config(text=self._path_str())

    def animate(self):
        if not self.playing: return
        self.draw()
        if self.step==0:
            self.lbl_step .config(text="START")
            self.lbl_ntype.config(text="OR node",  fg="#0055cc")
            self.lbl_path .config(text="P=[S]")
        else:
            m = self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
            self.lbl_step.config(text=f"Step {self.step}: {m}")
            if self.step%2==1:
                self.lbl_ntype.config(text="AND node", fg="#cc5500")
            else:
                self.lbl_ntype.config(text="OR node",  fg="#0055cc")
        self._update_sol()
        if self.step < len(self.path)-1:
            self.step+=1
            self.root.after(ANIM, self.animate)
        else:
            self.playing=False
            self.lbl_step.config(
                text="✓ SOLVED!" if self.path[-1]==GOAL else "✗ Stuck")

    def next_step(self):
        self.playing=False
        if not self.path or self.step>=len(self.path)-1: return
        self.step+=1
        self.draw()
        m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
        self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()


if __name__=="__main__":
    root=tk.Tk()
    App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
from collections import deque

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

def misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])

def apply_action(board, action):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    dr, dc = action
    nr, nc = r + dr, c + dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr * 3 + nc
        temp = list(board)
        temp[idx], temp[ni] = temp[ni], temp[idx]
        return tuple(temp)
    return None

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = {(-1, 0): "UP", (1, 0): "DOWN", (0, -1): "LEFT", (0, 1): "RIGHT"}

def sensorless_bfs(board1, board2):
    start_belief = frozenset([board1, board2])
    goal_belief = frozenset([GOAL])

    if start_belief == goal_belief:
        return [], [], 0
    queue = deque()
    queue.append((start_belief, [], [start_belief]))
    visited = {start_belief}
    nodes = 0

    while queue:
        belief, actions, belief_path = queue.popleft()
        nodes += 1

        for action in ACTIONS:
            new_belief_set = set()
            for board in belief:
                result = apply_action(board, action)
                if result is not None:
                    new_belief_set.add(result)
                else:
                    new_belief_set.add(board) 

            new_belief = frozenset(new_belief_set)
            if new_belief in visited:
                continue
            visited.add(new_belief)
            new_actions = actions + [action]
            new_path = belief_path + [new_belief]

            if new_belief == goal_belief or all(b == GOAL for b in new_belief):
                return new_actions, new_path, nodes
            queue.append((new_belief, new_actions, new_path))
            if nodes > 200000:
                return new_actions, new_path, nodes

    return [], [], nodes


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board1 = random_board()
        self.board2 = random_board()
        while self.board2 == self.board1:
            self.board2 = random_board()

        self.solution_actions = []
        self.belief_path = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)
        boards_frame = tk.Frame(left, bg="white")
        boards_frame.pack()

        b1_frame = tk.Frame(boards_frame, bg="white")
        b1_frame.pack(side="left", padx=6)
        tk.Label(b1_frame, text="Board 1", font=("Arial", 11, "bold"),
                 bg="white", fg="#0055aa").pack()
        self.canvas1 = tk.Canvas(
            b1_frame,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white", highlightthickness=0
        )
        self.canvas1.pack()
        self.lb_h1 = tk.Label(b1_frame, text="h(n) = —",
                               bg="white", fg="#c05000", font=("Arial", 10))
        self.lb_h1.pack(pady=2)

        b2_frame = tk.Frame(boards_frame, bg="white")
        b2_frame.pack(side="left", padx=6)
        tk.Label(b2_frame, text="Board 2", font=("Arial", 11, "bold"),
                 bg="white", fg="#aa5500").pack()
        self.canvas2 = tk.Canvas(
            b2_frame,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white", highlightthickness=0
        )
        self.canvas2.pack()
        self.lb_h2 = tk.Label(b2_frame, text="h(n) = —",
                               bg="white", fg="#c05000", font=("Arial", 10))
        self.lb_h2.pack(pady=2)

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=4)

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        info2 = tk.Frame(left, bg="white")
        info2.pack(pady=2)
        self.lb_belief = tk.Label(
            info2, text="Belief size: 2", bg="white", fg="#006400", font=("Arial", 10)
        )
        self.lb_belief.grid(row=0, column=0, padx=6)
        self.lb_visited = tk.Label(
            info2, text="Visited: 0", bg="white", fg="#00008b", font=("Arial", 10)
        )
        self.lb_visited.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Sensorless BFS)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_boards_at_step(self, step):
        if not self.belief_path:
            return self.board1, self.board2
        belief = self.belief_path[step]
        boards = list(belief)
        if len(boards) == 1:
            return boards[0], boards[0]
        return boards[0], boards[1]

    def draw_board(self, canvas, board, label_widget):
        canvas.delete("all")
        h_val = misplaced(board)
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE
            if val == 0:
                canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val), font=("Arial", 28),
                    fill="red" if wrong else "black"
                )
        label_widget.config(text=f"h(n) = {h_val}")

    def draw(self):
        if self.belief_path and self.step < len(self.belief_path):
            b1, b2 = self.get_boards_at_step(self.step)
        else:
            b1, b2 = self.board1, self.board2
        self.draw_board(self.canvas1, b1, self.lb_h1)
        self.draw_board(self.canvas2, b2, self.lb_h2)
        if self.belief_path and self.step < len(self.belief_path):
            self.lb_belief.config(text=f"Belief size: {len(self.belief_path[self.step])}")

    def shuffle(self):
        self.playing = False
        self.board1 = random_board()
        self.board2 = random_board()
        while self.board2 == self.board1:
            self.board2 = random_board()
        self.solution_actions = []
        self.belief_path = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board1 = GOAL
        self.board2 = GOAL
        self.solution_actions = []
        self.belief_path = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_h1.config(text="h(n) = —")
        self.lb_h2.config(text="h(n) = —")
        self.lb_belief.config(text="Belief size: 2")
        self.lb_visited.config(text="Visited: 0")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        actions, belief_path, nodes = sensorless_bfs(self.board1, self.board2)
        ms = (time.perf_counter() - t0) * 1000

        self.solution_actions = actions
        self.belief_path = belief_path
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(actions)}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_visited.config(text=f"Visited: {nodes}")

        if belief_path:
            final_belief = belief_path[-1]
            solved = all(b == GOAL for b in final_belief)
            self.step_label.config(text="✓ SOLVED" if solved else "✗ Not solved")
        else:
            self.step_label.config(text="✗ No solution")

        self.solution_box.delete("1.0", tk.END)
        if actions:
            self.playing = True
            self.animate()

    def _update_solution_box(self):
        if not self.belief_path or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            act = self.solution_actions[i - 1]
            name = ACTION_NAMES[act]
            belief = self.belief_path[i]
            h_vals = [misplaced(b) for b in belief]
            h_str = "/".join(str(h) for h in sorted(h_vals))
            self.solution_box.insert(tk.END, f"{i:2}. {name:<5} h={h_str}\n")
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        self.solution_box.tag_add("highlight", f"{self.step}.0", f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            act = self.solution_actions[self.step - 1]
            self.step_label.config(text=f"Step {self.step}: {ACTION_NAMES[act]}")

        self._update_solution_box()

        if self.step < len(self.belief_path) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            final_belief = self.belief_path[-1]
            solved = all(b == GOAL for b in final_belief)
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck")

    def next_step(self):
        self.playing = False
        if not self.belief_path:
            return
        if self.step < len(self.belief_path) - 1:
            self.step += 1
            self.draw()
            act = self.solution_actions[self.step - 1]
            self.step_label.config(text=f"Step {self.step}: {ACTION_NAMES[act]}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
from collections import deque

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 400

OBS_INDICES = (0, 1, 2)

def observe(board):
    return tuple(board[i] for i in OBS_INDICES)

def misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])

def apply_action(board, action):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    dr, dc = action
    nr, nc = r + dr, c + dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr * 3 + nc
        temp = list(board)
        temp[idx], temp[ni] = temp[ni], temp[idx]
        return tuple(temp)
    return None

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = {(-1, 0): "UP", (1, 0): "DOWN", (0, -1): "LEFT", (0, 1): "RIGHT"}

def get_all_reachable_boards():
    visited = {GOAL}
    queue = deque([GOAL])
    while queue:
        b = queue.popleft()
        for action in ACTIONS:
            nb = apply_action(b, action)
            if nb and nb not in visited:
                visited.add(nb)
                queue.append(nb)
    return visited

ALL_BOARDS = None

def get_all_boards():
    global ALL_BOARDS
    if ALL_BOARDS is None:
        ALL_BOARDS = get_all_reachable_boards()
    return ALL_BOARDS

def initial_belief(board):
    obs = observe(board)
    all_boards = get_all_boards()
    return frozenset(b for b in all_boards if observe(b) == obs)

def update_belief(belief, action, observation):
    new_belief = set()
    for board in belief:
        nb = apply_action(board, action)
        if nb is None:
            nb = board  
        if observe(nb) == observation:
            new_belief.add(nb)
    return frozenset(new_belief)

def partially_observable_bfs(true_board):
    start_obs = observe(true_board)
    start_belief = initial_belief(true_board)
    start = (true_board, start_belief)
    queue = deque()
    queue.append((true_board, start_belief, [true_board], [start_belief]))
    visited = {start}
    nodes = 0

    while queue:
        cur_board, cur_belief, path, bpath = queue.popleft()
        nodes += 1
        if cur_board == GOAL:
            return path, bpath, nodes
        for action in ACTIONS:
            nb = apply_action(cur_board, action)
            if nb is None:
                continue  
            obs = observe(nb)
            new_belief = update_belief(cur_belief, action, obs)
            if not new_belief:
                continue
            key = (nb, new_belief)
            if key in visited:
                continue
            visited.add(key)
            queue.append((nb, new_belief, path + [nb], bpath + [new_belief]))
            if nodes > 100000:
                return path, bpath, nodes
    return [], [], nodes


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Partially Observable BFS")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.root.after(100, self._precompute)
        self.board = random_board()
        self.path = []
        self.belief_path = []
        self.moves = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white", highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()
        obs_frame = tk.Frame(left, bg="white", bd=1, relief="groove")
        obs_frame.pack(pady=4, fill="x", padx=4)
        tk.Label(obs_frame, text="Observation (hàng 1):",
                 bg="white", font=("Arial", 9, "bold")).pack(side="left", padx=4)
        self.lb_obs = tk.Label(obs_frame, text="[?, ?, ?]",
                                bg="white", fg="#0055cc", font=("Consolas", 10))
        self.lb_obs.pack(side="left", padx=4)

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        info2 = tk.Frame(left, bg="white")
        info2.pack(pady=2)
        self.lb_belief = tk.Label(
            info2, text="Belief size: —", bg="white", fg="#006400", font=("Arial", 10)
        )
        self.lb_belief.grid(row=0, column=0, padx=6)
        self.lb_visited = tk.Label(
            info2, text="Visited: 0", bg="white", fg="#00008b", font=("Arial", 10)
        )
        self.lb_visited.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Partially Observable BFS)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=28, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def _precompute(self):
        self.step_label.config(text="Đang khởi tạo...")
        get_all_boards()
        self.step_label.config(text="Sẵn sàng")

    def current_board(self):
        if self.path and self.step < len(self.path):
            return self.path[self.step]
        return self.board

    def current_belief(self):
        if self.belief_path and self.step < len(self.belief_path):
            return self.belief_path[self.step]
        return None

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        belief = self.current_belief()
        h_val = misplaced(board)
        obs = observe(board)

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            is_observed = i in OBS_INDICES

            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                if is_observed:
                    fill = "#e0f0ff" if not wrong else "#ffe0e0"
                    outline_color = "#0055cc"
                    outline_w = 3
                else:
                    fill = "#f5f5f5" if not wrong else "#ffe0e0"
                    outline_color = "#999999"
                    outline_w = 2

                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill,
                    outline=outline_color, width=outline_w
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val), font=("Arial", 28),
                    fill="red" if wrong else ("black")
                )
            if not self.path and not is_observed and val != 0:
                self.canvas.create_text(
                    x2 - 12, y1 + 12,
                    text="?", font=("Arial", 10, "bold"), fill="#aaa"
                )

        self.lb_heuristic.config(text=f"h(n) = {h_val}")
        self.lb_obs.config(text=str(list(obs)))
        if belief is not None:
            self.lb_belief.config(text=f"Belief size: {len(belief)}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.path = []
        self.belief_path = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.path = []
        self.belief_path = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_belief.config(text="Belief size: —")
        self.lb_visited.config(text="Visited: 0")
        self.lb_obs.config(text="[?, ?, ?]")
        self.solution_box.delete("1.0", tk.END)

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def solve(self):
        if self.playing:
            return
        get_all_boards()

        t0 = time.perf_counter()
        path, belief_path, nodes = partially_observable_bfs(self.board)
        ms = (time.perf_counter() - t0) * 1000

        self.path = path
        self.belief_path = belief_path
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ] if len(path) > 1 else []
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1 if path else 0}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_visited.config(text=f"Visited: {nodes}")

        solved = bool(path) and path[-1] == GOAL
        self.step_label.config(text="✓ SOLVED" if solved else "✗ Not solved")

        self.solution_box.delete("1.0", tk.END)
        if path:
            self.playing = True
            self.animate()

    def _update_solution_box(self):
        if not self.path or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1] if i - 1 < len(self.moves) else "?"
            h = misplaced(self.path[i])
            belief = self.belief_path[i] if i < len(self.belief_path) else set()
            obs = observe(self.path[i])
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} h={h} |B|={len(belief)}\n"
                f"    obs={list(obs)}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        line = self.step * 2
        self.solution_box.tag_add("highlight", f"{line}.0", f"{line+1}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            m = self.moves[self.step - 1] if self.step - 1 < len(self.moves) else "?"
            self.step_label.config(text=f"Step {self.step}: {m}")

        self._update_solution_box()

        if self.step < len(self.path) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            solved = self.path[-1] == GOAL
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck")

    def next_step(self):
        self.playing = False
        if not self.path:
            return
        if self.step < len(self.path) - 1:
            self.step += 1
            self.draw()
            m = self.moves[self.step - 1] if self.step - 1 < len(self.moves) else "?"
            self.step_label.config(text=f"Step {self.step}: {m}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
import sys
sys.setrecursionlimit(200000)

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP  = 5
ANIM = 450

def misplaced(b):
    return sum(1 for i, v in enumerate(b) if v != 0 and v != GOAL[i])

def apply_action(b, a):
    idx = b.index(0)
    r, c = divmod(idx, 3)
    dr, dc = a
    nr, nc = r + dr, c + dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr * 3 + nc
        t = list(b); t[idx], t[ni] = t[ni], t[idx]
        return tuple(t)
    return None

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ANAMES  = {(-1, 0): "UP", (1, 0): "DOWN", (0, -1): "LEFT", (0, 1): "RIGHT"}

def backtracking_search(initial, max_depth=25):
    counter = [0]
    found   = [False]
    result  = [None]

    def bt(state, ancestor_set, path_s, path_a, depth):
        if found[0]:
            return
        counter[0] += 1
        if state == GOAL:
            found[0] = True
            result[0] = (list(path_s), list(path_a))
            return

        if depth == 0:
            return

        for a in ACTIONS:
            nb = apply_action(state, a)
            if nb is None:
                continue
            if nb in ancestor_set:
                continue
            ancestor_set.add(nb)
            path_s.append(nb)
            path_a.append(a)
            bt(nb, ancestor_set, path_s, path_a, depth - 1)
            if found[0]:
                return
            path_s.pop()
            path_a.pop()
            ancestor_set.discard(nb)

    bt(initial, {initial}, [initial], [], max_depth)

    if result[0]:
        ps, pa = result[0]
        return counter[0], ps, [ANAMES[a] for a in pa], len(ps) - 1
    return counter[0], [], [], 0

def is_solvable(b):
    arr = [x for x in b if x != 0]
    inv = sum(1 for i in range(len(arr)) for j in range(i+1, len(arr)) if arr[i] > arr[j])
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Pure Backtracking")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board   = random_board()
        self.path    = []
        self.moves   = []
        self.step    = 0
        self.playing = False

        main = tk.Frame(root, bg="white"); main.pack(padx=10, pady=10)
        left = tk.Frame(main, bg="white"); left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(left, width=3*TILE+4*GAP, height=3*TILE+4*GAP,
                                bg="white", highlightthickness=0)
        self.canvas.pack()

        self.lbl_step = tk.Label(left, text="", font=("Arial", 12), bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h = tk.Label(left, text="h(n) = —", bg="white",
                              fg="#c05000", font=("Arial", 10))
        self.lbl_h.pack()

        nf = tk.Frame(left, bg="white", bd=1, relief="groove")
        nf.pack(pady=4, fill="x", padx=4)
        tk.Label(nf, text="Trạng thái:", bg="white",
                 font=("Arial", 9, "bold")).pack(side="left", padx=4)
        self.lbl_ntype = tk.Label(nf, text="Chưa chạy", bg="white",
                                  fg="#0055cc", font=("Consolas", 10))
        self.lbl_ntype.pack(side="left", padx=4)

        r1 = tk.Frame(left, bg="white"); r1.pack(pady=2)
        self.lbl_nodes = tk.Label(r1, text="Nodes: —", bg="white"); self.lbl_nodes.grid(row=0, column=0, padx=5)
        self.lbl_steps = tk.Label(r1, text="Steps: —", bg="white"); self.lbl_steps.grid(row=0, column=1, padx=5)
        self.lbl_time  = tk.Label(r1, text="Time: — ms", bg="white"); self.lbl_time.grid(row=0, column=2, padx=5)

        r2 = tk.Frame(left, bg="white"); r2.pack(pady=2)
        self.lbl_depth = tk.Label(r2, text="Depth: —", bg="white",
                                  fg="#006400", font=("Arial", 10))
        self.lbl_depth.grid(row=0, column=0, padx=5)
        self.lbl_path = tk.Label(r2, text="P=[S]", bg="white",
                                 fg="#00008b", font=("Arial", 10))
        self.lbl_path.grid(row=0, column=1, padx=5)

        btns = tk.Frame(left, bg="white"); btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step ▶",  width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4)

        right = tk.Frame(main, bg="white"); right.pack(side="right", padx=10)
        tk.Label(right, text="Solution (Pure Backtracking)",
                 font=("Arial", 13, "bold"), bg="white").pack()
        self.sol = tk.Text(right, width=30, height=25, font=("Consolas", 9))
        self.sol.pack()
        self.draw()

    def cur_board(self):
        if self.path and self.step < len(self.path):
            return self.path[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        b = self.cur_board()
        self.lbl_h.config(text=f"h(n) = {misplaced(b)}")
        for i, val in enumerate(b):
            r, c = divmod(i, 3)
            x1 = GAP + c*(TILE+GAP); y1 = GAP + r*(TILE+GAP)
            x2 = x1 + TILE;          y2 = y1 + TILE
            if val == 0:
                self.canvas.create_rectangle(x1, y1, x2, y2, fill="#e8f5e9",
                    outline="#aaa", width=2, dash=(4, 3))
            else:
                w = val != GOAL[i]
                self.canvas.create_rectangle(x1, y1, x2, y2,
                    fill="#ffe0e0" if w else "#f5f5f5",
                    outline="#cc0000" if w else "#999", width=2)
                self.canvas.create_text((x1+x2)//2, (y1+y2)//2,
                    text=str(val), font=("Arial", 28),
                    fill="red" if w else "black")

    def _reset(self):
        self.path = []; self.moves = []; self.step = 0; self.playing = False
        self.lbl_nodes.config(text="Nodes: —")
        self.lbl_steps.config(text="Steps: —")
        self.lbl_time .config(text="Time: — ms")
        self.lbl_depth.config(text="Depth: —")
        self.lbl_path .config(text="P=[S]")
        self.lbl_step .config(text="")
        self.lbl_h    .config(text="h(n) = —")
        self.lbl_ntype.config(text="Chưa chạy", fg="#0055cc")
        self.sol.delete("1.0", tk.END)

    def shuffle(self):
        self.board = random_board(); self._reset(); self.draw()

    def reset(self):
        self.board = GOAL; self._reset(); self.draw()

    def _path_str(self):
        L = "SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        n = min(self.step + 1, len(L))
        return "P=[" + ",".join(L[i] for i in range(n)) + "]"

    def solve(self):
        if self.playing: return
        self._reset()
        self.draw()

        t0 = time.perf_counter()
        nodes, path_states, path_acts, depth = backtracking_search(self.board)
        ms = (time.perf_counter() - t0) * 1000

        self.path  = path_states
        self.moves = path_acts
        self.step  = 0

        solved = bool(path_states) and path_states[-1] == GOAL
        self.lbl_nodes.config(text=f"Nodes: {nodes}")
        self.lbl_steps.config(text=f"Steps: {depth}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_depth.config(text=f"Depth: {depth}")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed (tăng max_depth)")

        if path_states:
            self.playing = True
            self.animate()

    def _update_sol(self):
        if not self.path or self.step == 0: return
        self.sol.delete("1.0", tk.END)
        L = "SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        for i in range(1, self.step + 1):
            m   = self.moves[i-1] if i-1 < len(self.moves) else "?"
            nm  = L[i]   if i   < len(L) else f"N{i}"
            prv = L[i-1] if i-1 < len(L) else f"N{i-1}"
            self.sol.insert(tk.END,
                f"{i:2}. {prv} --{m}--> {nm}\n")
            self.sol.insert(tk.END,
                f"    ancestor_set add: {nm}\n")

        self.sol.tag_remove("hl", "1.0", tk.END)
        line = self.step * 2 - 1
        self.sol.tag_add("hl", f"{line}.0", f"{line+2}.end")
        self.sol.tag_config("hl", background="#cce5ff")
        self.sol.see(tk.END)
        self.lbl_path.config(text=self._path_str())

    def animate(self):
        if not self.playing: return
        self.draw()
        if self.step == 0:
            self.lbl_step .config(text="START")
            self.lbl_ntype.config(text="Khởi đầu", fg="#0055cc")
            self.lbl_path .config(text="P=[S]")
        else:
            m = self.moves[self.step-1] if self.step-1 < len(self.moves) else "?"
            self.lbl_step .config(text=f"Step {self.step}: {m}")
            self.lbl_ntype.config(text="Mở rộng → ancestor_set", fg="#cc5500")
        self._update_sol()
        if self.step < len(self.path) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            self.lbl_step.config(
                text="✓ SOLVED!" if self.path[-1] == GOAL else "✗ Stuck")
            self.lbl_ntype.config(text="Hoàn thành", fg="#006400")

    def next_step(self):
        self.playing = False
        if not self.path or self.step >= len(self.path) - 1: return
        self.step += 1
        self.draw()
        m = self.moves[self.step-1] if self.step-1 < len(self.moves) else "?"
        self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()


if __name__ == "__main__":
    root = tk.Tk()
    App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
import sys
sys.setrecursionlimit(200000)

GOAL  = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE  = 90
GAP   = 5
ANIM  = 500
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ANAMES  = {(-1,0):"UP",(1,0):"DOWN",(0,-1):"LEFT",(0,1):"RIGHT"}
MAX_DEPTH = 31

def apply_move(board, action):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    dr, dc = action
    nr, nc = r+dr, c+dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr*3+nc
        t  = list(board); t[idx], t[ni] = t[ni], t[idx]
        return tuple(t)
    return None

def misplaced(b):
    return sum(1 for i,v in enumerate(b) if v!=0 and v!=GOAL[i])

def is_solvable(b):
    arr = [x for x in b if x!=0]
    inv = sum(1 for i in range(len(arr))
              for j in range(i+1,len(arr)) if arr[i]>arr[j])
    return inv%2==0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t!=GOAL:
            return t

def fc_csp_search(initial, max_depth=MAX_DEPTH):
    nodes    = [0]
    found    = [False]
    sol_path = [None]
    log      = []

    def forward_checking(next_state, new_visited):
        return [d for d in ACTIONS
                if apply_move(next_state, d) is not None
                and apply_move(next_state, d) not in new_visited]

    def bt(depth, cur_state, path, visited):
        if found[0] or depth > max_depth:
            return
        nodes[0] += 1
        domain_cur = list(ACTIONS)  

        for action in ACTIONS:
            if found[0]: return
            aname      = ANAMES[action]
            next_state = apply_move(cur_state, action)

            if next_state is None:
                log.append({
                    "event"         : "INVALID",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : None,
                    "domain_current": domain_cur,
                    "domain_next"   : None,
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : "out of bounds → invalid value",
                })
                continue

            if next_state in visited:
                log.append({
                    "event"         : "CYCLE",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : next_state,
                    "domain_current": domain_cur,
                    "domain_next"   : None,
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : "next_state ∈ visited → cycle (no-repeat constraint violated)",
                })
                continue

            if next_state == GOAL:
                log.append({
                    "event"         : "GOAL",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : next_state,
                    "domain_current": domain_cur,
                    "domain_next"   : [],
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : "GOAL reached ✓",
                })
                found[0]    = True
                sol_path[0] = path+[action]
                return

            new_visited = visited | {next_state}
            domain_next = forward_checking(next_state, new_visited)
            fc_prune    = (len(domain_next) == 0)

            if fc_prune:
                log.append({
                    "event"         : "FC_PRUNE",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : next_state,
                    "domain_current": domain_cur,
                    "domain_next"   : [],
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : f"FC: domain(Move_{depth+2}) = ∅ → prune",
                })
                continue

            log.append({
                "event"         : "OK",
                "depth"         : depth,
                "state_before"  : cur_state,
                "action"        : action,
                "action_name"   : aname,
                "state_after"   : next_state,
                "domain_current": domain_cur,
                "domain_next"   : domain_next,
                "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                "reason"        : f"FC: domain(Move_{depth+2}) = {[ANAMES[d] for d in domain_next]}",
            })
            bt(depth+1, next_state, path+[action], new_visited)
            if found[0]: return

    bt(0, initial, [], {initial})
    return nodes[0], sol_path[0], log
EVENT_STYLE = {
    "INVALID"  : ("#eeeeee", "#aaaaaa"),
    "CYCLE"    : ("#fff3e0", "#e65100"),  
    "FC_PRUNE" : ("#ffcccc", "#cc0000"),
    "OK"       : ("#c8e6c9", "#1b5e20"),
    "GOAL"     : ("#d0eeff", "#0055cc"),
}

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — CSP + Forward Checking (AIMA-correct)")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board   = random_board()
        self.log     = []
        self.log_idx = 0
        self.playing = False
        self.solution= None

        main = tk.Frame(root, bg="white"); main.pack(padx=10, pady=10)
        left = tk.Frame(main, bg="white"); left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(left,
            width=3*TILE+4*GAP, height=3*TILE+4*GAP,
            bg="white", highlightthickness=0)
        self.canvas.pack()

        self.lbl_step = tk.Label(left, text="", font=("Arial",12), bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h = tk.Label(left, text="h(n) = —", bg="white",
                              fg="#c05000", font=("Arial",10))
        self.lbl_h.pack()

        nf = tk.Frame(left, bg="white", bd=1, relief="groove")
        nf.pack(pady=4, fill="x", padx=4)
        tk.Label(nf, text="Event:", bg="white",
                 font=("Arial",9,"bold")).pack(side="left", padx=4)
        self.lbl_ntype = tk.Label(nf, text="Not started", bg="white",
                                  fg="#0055cc", font=("Consolas",10))
        self.lbl_ntype.pack(side="left", padx=4)

        r1 = tk.Frame(left, bg="white"); r1.pack(pady=2)
        self.lbl_nodes = tk.Label(r1, text="Nodes: —",   bg="white")
        self.lbl_nodes.grid(row=0, column=0, padx=5)
        self.lbl_steps = tk.Label(r1, text="Moves: —",   bg="white")
        self.lbl_steps.grid(row=0, column=1, padx=5)
        self.lbl_time  = tk.Label(r1, text="Time: — ms", bg="white")
        self.lbl_time.grid(row=0, column=2, padx=5)

        r2 = tk.Frame(left, bg="white"); r2.pack(pady=2)
        self.lbl_depth = tk.Label(r2, text="Var: —", bg="white",
                                  fg="#006400", font=("Arial",10))
        self.lbl_depth.grid(row=0, column=0, padx=5)
        self.lbl_path  = tk.Label(r2, text="Path: []", bg="white",
                                  fg="#00008b", font=("Arial",9))
        self.lbl_path.grid(row=0, column=1, padx=5)

        btns = tk.Frame(left, bg="white"); btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10,
                  command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10,
                  command=self.reset).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10,
                  command=self.solve).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step ▶",  width=34,
                  command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4)

        right = tk.Frame(main, bg="white"); right.pack(side="right", padx=10)
        tk.Label(right, text="Solution (Forward Checking)",
                 font=("Arial",13,"bold"), bg="white").pack()

        fc_box = tk.Frame(right, bg="#f0f8ff", bd=1, relief="groove")
        fc_box.pack(fill="x", padx=4, pady=4)
        tk.Label(fc_box,
                 text="Basic FC (CSP · Variables = Moves · Pure AIMA):",
                 bg="#f0f8ff", font=("Arial",9,"bold")).pack(anchor="w", padx=4)

        self.lbl_fc_var    = tk.Label(fc_box, text="Var      : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_var.pack(anchor="w", padx=8)

        self.lbl_fc_assign = tk.Label(fc_box, text="Try      : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_assign.pack(anchor="w", padx=8)

        self.lbl_fc_consist= tk.Label(fc_box, text="① Consist: —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_consist.pack(anchor="w", padx=8)

        self.lbl_fc_domain = tk.Label(fc_box, text="② FC D+1 : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_domain.pack(anchor="w", padx=8)

        self.lbl_fc_result = tk.Label(fc_box, text="③ Result : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8,"bold"))
        self.lbl_fc_result.pack(anchor="w", padx=8, pady=(0,4))

        tk.Label(right, text="Domain[Move_i] = {UP, DOWN, LEFT, RIGHT}:",
                 bg="white", font=("Arial",9,"bold")).pack(anchor="w", padx=4)
        dgrid = tk.Frame(right, bg="white"); dgrid.pack(padx=4, pady=2)
        self.move_frames = {}
        for i, a in enumerate(ACTIONS):
            name = ANAMES[a]
            f   = tk.Frame(dgrid, bg="#f0f0f0", bd=1, relief="groove")
            f.grid(row=i//2, column=i%2, padx=4, pady=3)
            lbl = tk.Label(f, text=name, bg="#f0f0f0",
                           font=("Consolas",9,"bold"), width=14, height=2)
            lbl.pack()
            self.move_frames[name] = (f, lbl)

        self.sol = tk.Text(right, width=32, height=10, font=("Consolas",8))
        self.sol.pack(padx=4, pady=4)
        self.sol.tag_config("ok",     foreground="#006400")
        self.sol.tag_config("prune",  foreground="#cc0000")
        self.sol.tag_config("cycle",  foreground="#e65100")
        self.sol.tag_config("invalid",foreground="#aaaaaa")
        self.sol.tag_config("goal",   foreground="#0055cc",
                            font=("Consolas",8,"bold"))

        self._draw_board(self.board)
        self._reset_domain_grid()

    def _draw_board(self, board, moved_from=None):
        self.canvas.delete("all")
        self.lbl_h.config(text=f"h(n) = {misplaced(board)}")
        for i in range(9):
            r, c = divmod(i, 3)
            x1 = GAP+c*(TILE+GAP); y1 = GAP+r*(TILE+GAP)
            x2 = x1+TILE;          y2 = y1+TILE
            v  = board[i]
            if i == moved_from:
                fill, outline, dash = "#d0eeff", "#0055cc", ()
            elif v == 0:
                fill, outline, dash = "#e8f5e9", "#aaa",    (4,3)
            elif v == GOAL[i]:
                fill, outline, dash = "#f5f5f5", "#999",    ()
            else:
                fill, outline, dash = "#ffe0e0", "#cc0000", ()
            self.canvas.create_rectangle(x1,y1,x2,y2,
                fill=fill, outline=outline, width=2,
                dash=dash if dash else ())
            if v != 0:
                self.canvas.create_text((x1+x2)//2,(y1+y2)//2,
                    text=str(v), font=("Arial",28),
                    fill="#0055cc" if i==moved_from
                         else ("red" if v!=GOAL[i] else "black"))

    def _reset_domain_grid(self):
        for name, (f, lbl) in self.move_frames.items():
            lbl.config(text=f"{name}\n—", bg="#f0f0f0", fg="#555")
            f.config(bg="#f0f0f0")

    def _update_domain_grid(self, action, event, domain_next):
        chosen     = ANAMES[action] if action else None
        next_names = {ANAMES[a] for a in (domain_next or [])}

        for name, (f, lbl) in self.move_frames.items():
            if name == chosen:
                bg, fg = EVENT_STYLE[event]
                labels = {
                    "INVALID"  : "✗ out of bounds",
                    "CYCLE"    : "⊘ cycle → SKIP",
                    "FC_PRUNE" : "✗ FC PRUNE",
                    "OK"       : "✓ assign",
                    "GOAL"     : "★ GOAL",
                }
                note = labels[event]
            elif event in ("OK", "FC_PRUNE", "GOAL") and domain_next is not None:
                if name in next_names:
                    bg, fg, note = "#f5f5f5", "#333", "✓ in D+1"
                else:
                    bg, fg, note = "#fff3e0", "#e65100", "⊘ FC removed from D+1"
            else:
                bg, fg, note = "#f0f0f0", "#555", "in current D"

            lbl.config(text=f"{name}\n{note}", bg=bg, fg=fg)
            f.config(bg=bg)

    def _reset(self):
        self.log=[]; self.log_idx=0; self.playing=False; self.solution=None
        for w, t in [(self.lbl_nodes,"Nodes: —"),(self.lbl_steps,"Moves: —"),
                     (self.lbl_time,"Time: — ms"),(self.lbl_depth,"Var: —"),
                     (self.lbl_path,"Path: []"),(self.lbl_step,""),
                     (self.lbl_h,"h(n) = —")]:
            w.config(text=t)
        self.lbl_ntype.config(text="Not started", fg="#0055cc")
        for w, t in [(self.lbl_fc_var,"Var      : —"),
                     (self.lbl_fc_assign,"Try      : —"),
                     (self.lbl_fc_consist,"① Consist: —"),
                     (self.lbl_fc_domain,"② FC D+1 : —"),
                     (self.lbl_fc_result,"③ Result : —")]:
            w.config(text=t, fg="#333")
        self.sol.delete("1.0", tk.END)
        self._reset_domain_grid()

    def shuffle(self):
        self.board = random_board(); self._reset()
        self._draw_board(self.board)

    def reset(self):
        self.board = GOAL; self._reset()
        self._draw_board(self.board)

    def solve(self):
        if self.playing: return
        self._reset()
        self._draw_board(self.board)

        t0 = time.perf_counter()
        nodes, sol_path, log = fc_csp_search(self.board)
        ms = (time.perf_counter()-t0)*1000

        self.log      = log
        self.log_idx  = 0
        self.solution = sol_path

        solved     = (sol_path is not None)
        move_count = len(sol_path) if sol_path else 0

        self.lbl_nodes.config(text=f"Nodes: {nodes}")
        self.lbl_steps.config(text=f"Moves: {move_count}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed")

        if log:
            self.playing = True
            self.animate()

    def _show_entry(self, idx):
        if idx >= len(self.log): return
        e = self.log[idx]

        event   = e["event"]
        depth   = e["depth"]
        action  = e["action"]
        aname   = e["action_name"]
        sb      = e["state_before"]
        sa      = e["state_after"]
        d_next  = e["domain_next"]
        path_sf = e["path_so_far"]

        blank_from = sb.index(0)
        board_show = sa if (sa is not None and event not in ("INVALID","CYCLE")) else sb
        self._draw_board(board_show,
                         moved_from=blank_from if event not in ("INVALID","CYCLE") else None)
        self._update_domain_grid(action, event, d_next)

        self.lbl_depth.config(text=f"Var: Move_{depth+1}")
        path_short = path_sf[-5:]
        self.lbl_path.config(
            text="Path: [" + ", ".join(path_short) +
                 ("…" if len(path_sf)>5 else "") + "]")

        self.lbl_fc_var.config(text=f"Var      : Move_{depth+1}")
        self.lbl_fc_assign.config(text=f"Try      : Move_{depth+1} = {aname}")

        if event == "INVALID":
            self.lbl_fc_consist.config(
                text="① Consist: ✗  out of bounds", fg="#aaaaaa")
            self.lbl_fc_domain.config(
                text="② FC D+1 : (skipped — invalid value)", fg="#aaaaaa")
            self.lbl_fc_result.config(
                text="③ Result : SKIP — invalid value", fg="#aaaaaa")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → INVALID ✗")
            self.lbl_ntype.config(text="Invalid", fg="#aaaaaa")

        elif event == "CYCLE":
            self.lbl_fc_consist.config(
                text="① Consist: ✗  next ∈ visited", fg="#e65100")
            self.lbl_fc_domain.config(
                text="② FC D+1 : (skipped — no-repeat constraint violated)", fg="#aaaaaa")
            self.lbl_fc_result.config(
                text="③ Result : SKIP — cycle detected", fg="#e65100")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → CYCLE ⊘")
            self.lbl_ntype.config(text="Cycle (no-repeat)", fg="#e65100")

        elif event == "FC_PRUNE":
            self.lbl_fc_consist.config(
                text="① Consist: ✓  valid, not visited", fg="#006400")
            self.lbl_fc_domain.config(
                text="② FC D+1 : ∅  → PRUNE", fg="#cc0000")
            self.lbl_fc_result.config(
                text="③ Result : FC PRUNE → backtrack", fg="#cc0000")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → FC PRUNE ✗")
            self.lbl_ntype.config(text="FC Prune!", fg="#cc0000")

        elif event == "GOAL":
            self.lbl_fc_consist.config(
                text="① Consist: ✓  valid, not visited", fg="#006400")
            self.lbl_fc_domain.config(
                text="② FC D+1 : GOAL → FC not needed", fg="#0055cc")
            self.lbl_fc_result.config(
                text="③ Result : GOAL ✓", fg="#0055cc")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → GOAL ✓")
            self.lbl_ntype.config(text="GOAL ✓", fg="#0055cc")

        else: 
            next_names = [ANAMES[a] for a in d_next] if d_next else []
            self.lbl_fc_consist.config(
                text="① Consist: ✓  valid, not visited", fg="#006400")
            self.lbl_fc_domain.config(
                text=f"② FC D+1 : {{{', '.join(next_names)}}}", fg="#006400")
            self.lbl_fc_result.config(
                text=f"③ Result : OK → assign Move_{depth+1} + recurse",
                fg="#006400")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → OK ✓")
            self.lbl_ntype.config(text="Assign var", fg="#cc5500")

        self.sol.delete("1.0", tk.END)
        for i, en in enumerate(self.log[:idx+1]):
            p, an, ev = en["depth"], en["action_name"], en["event"]
            line = f"{i+1:3}. Move_{p+1}={an:<5}  "
            if ev == "GOAL":
                self.sol.insert(tk.END, line+"GOAL ✓\n",       "goal")
            elif ev == "FC_PRUNE":
                self.sol.insert(tk.END, line+"FC_PRUNE ✗\n",   "prune")
            elif ev == "CYCLE":
                self.sol.insert(tk.END, line+"CYCLE ⊘\n",      "cycle")
            elif ev == "INVALID":
                self.sol.insert(tk.END, line+"INVALID ✗\n",    "invalid")
            else:
                dn = [ANAMES[a] for a in en["domain_next"]] if en["domain_next"] else []
                self.sol.insert(tk.END, line+f"ok d+1={dn}\n", "ok")
        self.sol.see(tk.END)

    def animate(self):
        if not self.playing: return
        self._show_entry(self.log_idx)
        if self.log_idx < len(self.log)-1:
            self.log_idx += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            if self.solution:
                self._draw_board(GOAL)
                self.lbl_step .config(text="✓ SOLVED!")
                self.lbl_ntype.config(text="Done", fg="#006400")
            else:
                self.lbl_step .config(text="✗ No solution found")
                self.lbl_ntype.config(text="Failed", fg="#cc0000")

    def next_step(self):
        self.playing = False
        if not self.log: return
        if self.log_idx < len(self.log)-1:
            self.log_idx += 1
        self._show_entry(self.log_idx)


if __name__ == "__main__":
    root = tk.Tk()
    App(root)
    root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
from collections import deque

GOAL   = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE   = 90
GAP    = 5
ANIM   = 500

ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ANAMES  = {(-1,0):"UP",(1,0):"DOWN",(0,-1):"LEFT",(0,1):"RIGHT"}

def misplaced(b):
    return sum(1 for i,v in enumerate(b) if v!=0 and v!=GOAL[i])

def apply_action(b, a):
    idx=b.index(0); r,c=divmod(idx,3); dr,dc=a; nr,nc=r+dr,c+dc
    if 0<=nr<3 and 0<=nc<3:
        ni=nr*3+nc; t=list(b); t[idx],t[ni]=t[ni],t[idx]; return tuple(t)
    return None

VARIABLES = list(range(9))
def binary_constraint(xi, vi, xj, vj):
    return vi != vj

def unary_constraint(xi, vi):
    return vi == GOAL[xi]
def ac3(domains, log=None):
    queue = deque((xi, xj)
                  for xi in VARIABLES
                  for xj in VARIABLES if xi != xj)
    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj, log):
            if len(domains[xi]) == 0:
                return False          
            for xk in VARIABLES:
                if xk != xj:
                    queue.append((xk, xi))
    return True

def revise(domains, xi, xj, log=None):
    revised   = False
    to_remove = set()
    for vx in domains[xi]:
        if not any(binary_constraint(xi, vx, xj, vy) for vy in domains[xj]):
            to_remove.add(vx)
            if log is not None:
                log.append((xi, xj, vx))
    if to_remove:
        domains[xi] -= to_remove
        revised = True
    return revised
def backtracking_search(initial_domains, log_ac3, counter):
    return backtrack({}, initial_domains, log_ac3, counter)

def backtrack(assignment, domains, log_ac3, counter):
    counter[0] += 1
    if len(assignment) == 9:
        return dict(assignment)
    xi = next(v for v in VARIABLES if v not in assignment)

    for vi in sorted(domains[xi]):
        if not unary_constraint(xi, vi):
            continue
        if not all(binary_constraint(xi, vi, xj, assignment[xj])
                   for xj in assignment):
            continue
        assignment[xi] = vi
        domains_copy = {v: set(d) for v, d in domains.items()}
        domains_copy[xi] = {vi}
        inferences_ok = ac3(domains_copy, log_ac3)

        if inferences_ok:
            result = backtrack(assignment, domains_copy, log_ac3, counter)
            if result is not None:
                return result

        del assignment[xi]

    return None   
def bfs_path(start):
    parent  = {start: None}
    act_map = {start: None}
    q = deque([start])
    while q:
        s = q.popleft()
        if s == GOAL:
            path, acts = [], []
            while act_map[s] is not None:
                acts.append(ANAMES[act_map[s]]); path.append(s); s = parent[s]
            path.append(start); path.reverse(); acts.reverse()
            return path, acts
        for a in ACTIONS:
            nb = apply_action(s, a)
            if nb and nb not in parent:
                parent[nb]=s; act_map[nb]=a; q.append(nb)
    return [start], []

def csp_ac3_solve(board):
    counter  = [0]
    log_ac3  = []
    domains = {i: {GOAL[i]} for i in VARIABLES}
    ok = ac3(domains, log_ac3)
    if not ok:
        return [], 0, [], 0, log_ac3
    assignment = backtracking_search(
        {v: set(d) for v, d in domains.items()}, log_ac3, counter)

    if assignment is None:
        return [], counter[0], [], 0, log_ac3

    solved_board = tuple(assignment[i] for i in range(9))
    if solved_board != GOAL:
        return [], counter[0], [], 0, log_ac3

    path_states, path_acts = bfs_path(board)
    return path_states, counter[0], path_acts, len(path_states)-1, log_ac3

def is_solvable(b):
    arr=[x for x in b if x!=0]
    inv=sum(1 for i in range(len(arr)) for j in range(i+1,len(arr)) if arr[i]>arr[j])
    return inv%2==0

def random_board():
    b=list(GOAL)
    while True:
        random.shuffle(b); t=tuple(b)
        if is_solvable(t) and t!=GOAL: return t

class App:
    def __init__(self, root):
        self.root=root
        self.root.title("8 Puzzle — CSP: Backtracking + AC-3 (MAC)")
        self.root.configure(bg="white")
        self.root.resizable(False,False)

        self.board  =random_board()
        self.path   =[]
        self.moves  =[]
        self.log_ac3=[]
        self.step   =0
        self.playing=False

        main=tk.Frame(root,bg="white"); main.pack(padx=10,pady=10)
        left=tk.Frame(main,bg="white"); left.pack(side="left",padx=10)

        self.canvas=tk.Canvas(left,width=3*TILE+4*GAP,height=3*TILE+4*GAP,
                              bg="white",highlightthickness=0)
        self.canvas.pack()

        self.lbl_step=tk.Label(left,text="",font=("Arial",12),bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h=tk.Label(left,text="h(n) = —",bg="white",
                            fg="#c05000",font=("Arial",10))
        self.lbl_h.pack()

        nf=tk.Frame(left,bg="white",bd=1,relief="groove")
        nf.pack(pady=4,fill="x",padx=4)
        tk.Label(nf,text="Phase:",bg="white",
                 font=("Arial",9,"bold")).pack(side="left",padx=4)
        self.lbl_ntype=tk.Label(nf,text="—",bg="white",
                                fg="#0055cc",font=("Consolas",10))
        self.lbl_ntype.pack(side="left",padx=4)

        r1=tk.Frame(left,bg="white"); r1.pack(pady=2)
        self.lbl_nodes=tk.Label(r1,text="Nodes: —",bg="white"); self.lbl_nodes.grid(row=0,column=0,padx=5)
        self.lbl_steps=tk.Label(r1,text="Steps: —",bg="white"); self.lbl_steps.grid(row=0,column=1,padx=5)
        self.lbl_time =tk.Label(r1,text="Time: — ms",bg="white"); self.lbl_time.grid(row=0,column=2,padx=5)

        r2=tk.Frame(left,bg="white"); r2.pack(pady=2)
        self.lbl_depth=tk.Label(r2,text="Depth: —",bg="white",
                                fg="#006400",font=("Arial",10))
        self.lbl_depth.grid(row=0,column=0,padx=5)
        self.lbl_path=tk.Label(r2,text="P=[S]",bg="white",
                               fg="#00008b",font=("Arial",10))
        self.lbl_path.grid(row=0,column=1,padx=5)

        btns=tk.Frame(left,bg="white"); btns.pack(pady=6)
        tk.Button(btns,text="Shuffle",width=10,command=self.shuffle).grid(row=0,column=0,padx=4)
        tk.Button(btns,text="Reset",  width=10,command=self.reset  ).grid(row=0,column=1,padx=4)
        tk.Button(btns,text="Solve",  width=10,command=self.solve  ).grid(row=0,column=2,padx=4)
        tk.Button(btns,text="Step ▶",width=34,command=self.next_step).grid(
            row=1,column=0,columnspan=3,pady=4)

        right=tk.Frame(main,bg="white"); right.pack(side="right",padx=10)
        tk.Label(right,text="Solution (AC-3 + Backtracking)",
                 font=("Arial",13,"bold"),bg="white").pack()
        self.sol=tk.Text(right,width=30,height=25,font=("Consolas",9))
        self.sol.pack()
        self.draw()
    def cur_board(self):
        if self.path and self.step<len(self.path): return self.path[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        b=self.cur_board()
        self.lbl_h.config(text=f"h(n) = {misplaced(b)}")
        for i,val in enumerate(b):
            r,c=divmod(i,3)
            x1=GAP+c*(TILE+GAP); y1=GAP+r*(TILE+GAP)
            x2=x1+TILE;          y2=y1+TILE
            if val==0:
                self.canvas.create_rectangle(x1,y1,x2,y2,fill="#e8f5e9",
                    outline="#aaa",width=2,dash=(4,3))
            else:
                wrong=val!=GOAL[i]
                self.canvas.create_rectangle(x1,y1,x2,y2,
                    fill="#ffe0e0" if wrong else "#f5f5f5",
                    outline="#cc0000" if wrong else "#999",width=2)
                self.canvas.create_text((x1+x2)//2,(y1+y2)//2,
                    text=str(val),font=("Arial",28),
                    fill="red" if wrong else "black")

    def _reset(self):
        self.path=[];self.moves=[];self.log_ac3=[]
        self.step=0;self.playing=False
        for lbl,txt in [(self.lbl_nodes,"Nodes: —"),(self.lbl_steps,"Steps: —"),
                        (self.lbl_time,"Time: — ms"),(self.lbl_depth,"Depth: —"),
                        (self.lbl_path,"P=[S]"),(self.lbl_step,""),(self.lbl_h,"h(n) = —")]:
            lbl.config(text=txt)
        self.lbl_ntype.config(text="—",fg="#0055cc")
        self.sol.delete("1.0",tk.END)

    def shuffle(self): self.board=random_board();self._reset();self.draw()
    def reset(self):   self.board=GOAL;self._reset();self.draw()

    def _path_str(self):
        L="SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        n=min(self.step+1,len(L))
        return "P=["+ ",".join(L[i] for i in range(n))+"]"

    def solve(self):
        if self.playing: return
        self._reset(); self.draw()
        t0=time.perf_counter()
        path_states,nodes,path_acts,depth,log_ac3=csp_ac3_solve(self.board)
        ms=(time.perf_counter()-t0)*1000

        self.path=path_states; self.moves=path_acts
        self.log_ac3=log_ac3; self.step=0

        solved=bool(path_states) and path_states[-1]==GOAL
        self.lbl_nodes.config(text=f"Nodes: {nodes}")
        self.lbl_steps.config(text=f"Steps: {depth}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_depth.config(text=f"Depth: {depth}")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed")

        self.sol.delete("1.0",tk.END)
        self.sol.insert(tk.END,"=== AC-3 Arc Revisions ===\n")
        if log_ac3:
            for k,(xi,xj,rv) in enumerate(log_ac3):
                self.sol.insert(tk.END,f"  revise(X{xi},X{xj}): rm {rv}\n")
                if k>=14:
                    self.sol.insert(tk.END,f"  ...({len(log_ac3)} total)\n"); break
        else:
            self.sol.insert(tk.END,"  (no values removed — domains already\n")
            self.sol.insert(tk.END,"   arc-consistent after unary pruning)\n")
        self.sol.insert(tk.END,"\n=== CSP Assignment ===\n")
        self.sol.insert(tk.END,"  Xi = GOAL[i] for i in 0..8\n")
        self.sol.insert(tk.END,f"  Result: {GOAL}\n")
        self.sol.insert(tk.END,"\n=== Move Sequence (BFS) ===\n")

        if path_states:
            self.playing=True
            self.lbl_ntype.config(text="Backtracking + MAC",fg="#006600")
            self.animate()

    def _update_sol(self):
        if not self.path or self.step==0: return
        content=self.sol.get("1.0",tk.END)
        marker="=== Move Sequence (BFS) ===\n"
        idx=content.find(marker)
        if idx>=0:
            lb=content[:idx+len(marker)].count("\n")
            self.sol.delete(f"{lb+1}.0",tk.END)
        L="SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        for i in range(1,self.step+1):
            m=self.moves[i-1] if i-1<len(self.moves) else "?"
            h=misplaced(self.path[i])
            nm=L[i]   if i  <len(L) else f"N{i}"
            pv=L[i-1] if i-1<len(L) else f"N{i-1}"
            self.sol.insert(tk.END,f"{i:2}. {pv} →{m:<6} h={h}  [{nm}]\n")
        self.sol.tag_remove("hl","1.0",tk.END)
        total=int(self.sol.index("end-1c").split(".")[0])
        self.sol.tag_add("hl",f"{total}.0",f"{total}.end")
        self.sol.tag_config("hl",background="#cce5ff")
        self.sol.see(tk.END)
        self.lbl_path.config(text=self._path_str())

    def animate(self):
        if not self.playing: return
        self.draw()
        if self.step==0:
            self.lbl_step.config(text="START"); self.lbl_path.config(text="P=[S]")
        else:
            m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
            self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()
        if self.step<len(self.path)-1:
            self.step+=1; self.root.after(ANIM,self.animate)
        else:
            self.playing=False
            self.lbl_step.config(text="✓ SOLVED!" if self.path[-1]==GOAL else "✗ Stuck")

    def next_step(self):
        self.playing=False
        if not self.path or self.step>=len(self.path)-1: return
        self.step+=1; self.draw()
        m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
        self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()

if __name__=="__main__":
    root=tk.Tk(); App(root); root.mainloop()

In [ ]:
import tkinter as tk
import random
import time
from collections import deque
GOAL   = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE   = 90
GAP    = 5
ANIM   = 500

ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ANAMES  = {(-1,0):"UP",(1,0):"DOWN",(0,-1):"LEFT",(0,1):"RIGHT"}

VARIABLES = list(range(9))
DOMAIN    = list(range(9))

def misplaced(b):
    return sum(1 for i,v in enumerate(b) if v!=0 and v!=GOAL[i])

def apply_action(b, a):
    idx=b.index(0); r,c=divmod(idx,3); dr,dc=a; nr,nc=r+dr,c+dc
    if 0<=nr<3 and 0<=nc<3:
        ni=nr*3+nc; t=list(b); t[idx],t[ni]=t[ni],t[idx]; return tuple(t)
    return None

def count_conflicts(var, value, assignment):
    c = 0
    for j in VARIABLES:
        if j != var and assignment[j] == value:
            c += 1
    if value != GOAL[var]:
        c += 1
    return c

def conflicted_variables(assignment):
    conflicted = []
    for i in VARIABLES:
        if count_conflicts(i, assignment[i], assignment) > 0:
            conflicted.append(i)
    return conflicted

def min_conflicts(max_steps=100_000):
    values = list(range(9))
    random.shuffle(values)
    assignment = {i: values[i] for i in VARIABLES}
    log = []  
    for step in range(1, max_steps + 1):
        conflicted = conflicted_variables(assignment)
        if not conflicted:
            return assignment, step, log   
        var = random.choice(conflicted)
        conflict_counts = [
            (count_conflicts(var, v, assignment), v)
            for v in DOMAIN
        ]
        min_c = min(c for c, _ in conflict_counts)
        best_values = [v for c, v in conflict_counts if c == min_c]
        chosen = random.choice(best_values)
        log.append((step, var, assignment[var], chosen, min_c))
        assignment[var] = chosen

    return None, max_steps, log 
def bfs_path(start):
    parent  = {start: None}
    act_map = {start: None}
    q = deque([start])
    while q:
        s = q.popleft()
        if s == GOAL:
            path, acts = [], []
            while act_map[s] is not None:
                acts.append(ANAMES[act_map[s]]); path.append(s); s = parent[s]
            path.append(start); path.reverse(); acts.reverse()
            return path, acts
        for a in ACTIONS:
            nb = apply_action(s, a)
            if nb and nb not in parent:
                parent[nb]=s; act_map[nb]=a; q.append(nb)
    return [start], []

def minconflicts_solve(board, max_steps=100_000):
    assignment, steps, log = min_conflicts(max_steps)

    if assignment is None:
        return [], steps, [], 0, log

    solved_board = tuple(assignment[i] for i in range(9))
    if solved_board != GOAL:
        return [], steps, [], 0, log

    path_states, path_acts = bfs_path(board)
    return path_states, steps, path_acts, len(path_states)-1, log
def is_solvable(b):
    arr=[x for x in b if x!=0]
    inv=sum(1 for i in range(len(arr)) for j in range(i+1,len(arr)) if arr[i]>arr[j])
    return inv%2==0

def random_board():
    b=list(GOAL)
    while True:
        random.shuffle(b); t=tuple(b)
        if is_solvable(t) and t!=GOAL: return t

class App:
    def __init__(self, root):
        self.root=root
        self.root.title("8 Puzzle — CSP: Min-Conflicts ")
        self.root.configure(bg="white")
        self.root.resizable(False,False)

        self.board  =random_board()
        self.path   =[]
        self.moves  =[]
        self.log    =[]
        self.step   =0
        self.playing=False

        main=tk.Frame(root,bg="white"); main.pack(padx=10,pady=10)
        left=tk.Frame(main,bg="white"); left.pack(side="left",padx=10)

        self.canvas=tk.Canvas(left,width=3*TILE+4*GAP,height=3*TILE+4*GAP,
                              bg="white",highlightthickness=0)
        self.canvas.pack()

        self.lbl_step=tk.Label(left,text="",font=("Arial",12),bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h=tk.Label(left,text="h(n) = —",bg="white",
                            fg="#c05000",font=("Arial",10))
        self.lbl_h.pack()

        nf=tk.Frame(left,bg="white",bd=1,relief="groove")
        nf.pack(pady=4,fill="x",padx=4)
        tk.Label(nf,text="Phase:",bg="white",
                 font=("Arial",9,"bold")).pack(side="left",padx=4)
        self.lbl_ntype=tk.Label(nf,text="—",bg="white",
                                fg="#0055cc",font=("Consolas",10))
        self.lbl_ntype.pack(side="left",padx=4)

        r1=tk.Frame(left,bg="white"); r1.pack(pady=2)
        self.lbl_nodes=tk.Label(r1,text="Steps: —",bg="white")
        self.lbl_nodes.grid(row=0,column=0,padx=5)
        self.lbl_steps=tk.Label(r1,text="Moves: —",bg="white")
        self.lbl_steps.grid(row=0,column=1,padx=5)
        self.lbl_time =tk.Label(r1,text="Time: — ms",bg="white")
        self.lbl_time.grid(row=0,column=2,padx=5)

        r2=tk.Frame(left,bg="white"); r2.pack(pady=2)
        self.lbl_depth=tk.Label(r2,text="Depth: —",bg="white",
                                fg="#006400",font=("Arial",10))
        self.lbl_depth.grid(row=0,column=0,padx=5)
        self.lbl_path=tk.Label(r2,text="P=[S]",bg="white",
                               fg="#00008b",font=("Arial",10))
        self.lbl_path.grid(row=0,column=1,padx=5)

        btns=tk.Frame(left,bg="white"); btns.pack(pady=6)
        tk.Button(btns,text="Shuffle",width=10,command=self.shuffle).grid(row=0,column=0,padx=4)
        tk.Button(btns,text="Reset",  width=10,command=self.reset  ).grid(row=0,column=1,padx=4)
        tk.Button(btns,text="Solve",  width=10,command=self.solve  ).grid(row=0,column=2,padx=4)
        tk.Button(btns,text="Step ▶",width=34,command=self.next_step).grid(
            row=1,column=0,columnspan=3,pady=4)

        right=tk.Frame(main,bg="white"); right.pack(side="right",padx=10)
        tk.Label(right,text="Solution (Min-Conflicts)",
                 font=("Arial",13,"bold"),bg="white").pack()
        self.sol=tk.Text(right,width=32,height=25,font=("Consolas",9))
        self.sol.pack()
        self.draw()

    def cur_board(self):
        if self.path and self.step<len(self.path): return self.path[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        b=self.cur_board()
        self.lbl_h.config(text=f"h(n) = {misplaced(b)}")
        for i,val in enumerate(b):
            r,c=divmod(i,3)
            x1=GAP+c*(TILE+GAP); y1=GAP+r*(TILE+GAP)
            x2=x1+TILE;          y2=y1+TILE
            if val==0:
                self.canvas.create_rectangle(x1,y1,x2,y2,fill="#e8f5e9",
                    outline="#aaa",width=2,dash=(4,3))
            else:
                wrong=val!=GOAL[i]
                self.canvas.create_rectangle(x1,y1,x2,y2,
                    fill="#ffe0e0" if wrong else "#f5f5f5",
                    outline="#cc0000" if wrong else "#999",width=2)
                self.canvas.create_text((x1+x2)//2,(y1+y2)//2,
                    text=str(val),font=("Arial",28),
                    fill="red" if wrong else "black")

    def _reset(self):
        self.path=[];self.moves=[];self.log=[]
        self.step=0;self.playing=False
        for lbl,txt in [(self.lbl_nodes,"Steps: —"),(self.lbl_steps,"Moves: —"),
                        (self.lbl_time,"Time: — ms"),(self.lbl_depth,"Depth: —"),
                        (self.lbl_path,"P=[S]"),(self.lbl_step,""),(self.lbl_h,"h(n) = —")]:
            lbl.config(text=txt)
        self.lbl_ntype.config(text="—",fg="#0055cc")
        self.sol.delete("1.0",tk.END)

    def shuffle(self): self.board=random_board();self._reset();self.draw()
    def reset(self):   self.board=GOAL;self._reset();self.draw()

    def _path_str(self):
        L="SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        n=min(self.step+1,len(L))
        return "P=["+ ",".join(L[i] for i in range(n))+"]"

    def solve(self):
        if self.playing: return
        self._reset(); self.draw()
        t0=time.perf_counter()
        path_states,csp_steps,path_acts,depth,log=minconflicts_solve(self.board)
        ms=(time.perf_counter()-t0)*1000

        self.path=path_states; self.moves=path_acts
        self.log=log; self.step=0

        solved=bool(path_states) and path_states[-1]==GOAL
        self.lbl_nodes.config(text=f"Steps: {csp_steps}")
        self.lbl_steps.config(text=f"Moves: {depth}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_depth.config(text=f"Depth: {depth}")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed")
        self.sol.delete("1.0",tk.END)
        self.sol.insert(tk.END,"=== Min-Conflicts Iterations ===\n")
        self.sol.insert(tk.END,f"  Total CSP steps: {csp_steps}\n\n")
        self.sol.insert(tk.END,"  step  var  old→new  min_c\n")
        self.sol.insert(tk.END,"  " + "─"*28 + "\n")
        shown = log[:15]
        for (step,var,old,new,mc) in shown:
            self.sol.insert(tk.END,
                f"  {step:4d}  X{var}   {old}→{new}      c={mc}\n")
        if len(log)>15:
            self.sol.insert(tk.END,f"  ...({len(log)} total iterations)\n")
        self.sol.insert(tk.END,"\n=== Move Sequence (BFS) ===\n")

        if path_states:
            self.playing=True
            self.lbl_ntype.config(text="Min-Conflicts",fg="#006600")
            self.animate()

    def _update_sol(self):
        if not self.path or self.step==0: return
        content=self.sol.get("1.0",tk.END)
        marker="=== Move Sequence (BFS) ===\n"
        idx=content.find(marker)
        if idx>=0:
            lb=content[:idx+len(marker)].count("\n")
            self.sol.delete(f"{lb+1}.0",tk.END)
        L="SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        for i in range(1,self.step+1):
            m=self.moves[i-1] if i-1<len(self.moves) else "?"
            h=misplaced(self.path[i])
            nm=L[i]   if i  <len(L) else f"N{i}"
            pv=L[i-1] if i-1<len(L) else f"N{i-1}"
            self.sol.insert(tk.END,f"{i:2}. {pv} →{m:<6} h={h}  [{nm}]\n")
        self.sol.tag_remove("hl","1.0",tk.END)
        total=int(self.sol.index("end-1c").split(".")[0])
        self.sol.tag_add("hl",f"{total}.0",f"{total}.end")
        self.sol.tag_config("hl",background="#cce5ff")
        self.sol.see(tk.END)
        self.lbl_path.config(text=self._path_str())

    def animate(self):
        if not self.playing: return
        self.draw()
        if self.step==0:
            self.lbl_step.config(text="START"); self.lbl_path.config(text="P=[S]")
        else:
            m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
            self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()
        if self.step<len(self.path)-1:
            self.step+=1; self.root.after(ANIM,self.animate)
        else:
            self.playing=False
            self.lbl_step.config(text="✓ SOLVED!" if self.path[-1]==GOAL else "✗ Stuck")

    def next_step(self):
        self.playing=False
        if not self.path or self.step>=len(self.path)-1: return
        self.step+=1; self.draw()
        m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
        self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()

if __name__=="__main__":
    root=tk.Tk(); App(root); root.mainloop()

In [ ]:
import random

RULE_TABLE = {
    (0,0): ["DOWN", "RIGHT"],
    (0,1): ["DOWN", "LEFT", "RIGHT"],
    (0,2): ["DOWN", "LEFT"],
    (1,0): ["UP", "DOWN", "RIGHT"],
    (1,1): ["UP", "DOWN", "LEFT", "RIGHT"],
    (1,2): ["UP", "DOWN", "LEFT"],
    (2,0): ["UP", "RIGHT"],
    (2,1): ["UP", "LEFT", "RIGHT"],
    (2,2): ["UP", "LEFT"],
}
def sensor(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return (i, j)
def processor(blank_pos):
    valid = RULE_TABLE[blank_pos]
    print(f"Blank tại {blank_pos}")
    return random.choice(valid)
def actuator(move):
    print(f"Quyết định: di chuyển blank sang {move}")
state = []
print("Nhập ma trận 3x3:")
for i in range(3):
    row = list(map(int, input(f"Hàng {i+1}: ").split()))
    state.append(row)
print()
blank_pos = sensor(state)
move= processor(blank_pos)
actuator(move)

In [ ]:
import random

GOAL = [[1,2,3],[4,5,6],[7,8,0]]

LUAT = {
    (0,0): ["DOWN", "RIGHT"],
    (0,1): ["DOWN", "LEFT", "RIGHT"],
    (0,2): ["DOWN", "LEFT"],
    (1,0): ["UP", "DOWN", "RIGHT"],
    (1,1): ["UP", "DOWN", "LEFT", "RIGHT"],
    (1,2): ["UP", "DOWN", "LEFT"],
    (2,0): ["UP", "RIGHT"],
    (2,1): ["UP", "LEFT", "RIGHT"],
    (2,2): ["UP", "LEFT"],
}

DELTA = {"UP":(-1,0), "DOWN":(1,0), "LEFT":(0,-1), "RIGHT":(0,1)}
def sensor(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return (i, j)

def processor(blank):
    r, c = blank
    if r == 0 and c == 0:
        huong = random.choice(["DOWN", "RIGHT"])
    elif r == 0 and c == 1:
        huong = random.choice(["DOWN", "LEFT", "RIGHT"])
    elif r == 0 and c == 2:
        huong = random.choice(["DOWN", "LEFT"])
    elif r == 1 and c == 0:
        huong = random.choice(["UP", "DOWN", "RIGHT"])
    elif r == 1 and c == 1:
        huong = random.choice(["UP", "DOWN", "LEFT", "RIGHT"])
    elif r == 1 and c == 2:
        huong = random.choice(["UP", "DOWN", "LEFT"])
    elif r == 2 and c == 0:
        huong = random.choice(["UP", "RIGHT"])
    elif r == 2 and c == 1:
        huong = random.choice(["UP", "LEFT", "RIGHT"])
    else:
        huong = random.choice(["UP", "LEFT"])
    return huong

def actuator(state, blank, huong):
    r, c = blank
    moi = [hang[:] for hang in state]
 
    if huong == "UP":
        moi[r][c], moi[r-1][c] = moi[r-1][c], moi[r][c]
    elif huong == "DOWN":
        moi[r][c], moi[r+1][c] = moi[r+1][c], moi[r][c]
    elif huong == "LEFT":
        moi[r][c], moi[r][c-1] = moi[r][c-1], moi[r][c]
    elif huong == "RIGHT":
        moi[r][c], moi[r][c+1] = moi[r][c+1], moi[r][c]
 
    print(f"Da di chuyen blank sang {huong}")
    return moi
print("Ma tran ban dau:")
display(state)
state = [row[:] for row in GOAL]
nums = list(range(9))
random.shuffle(nums)

state = [nums[:3],
         nums[3:6],
         nums[6:]]
step = 0
lichsu={}
while True:
    blank = sensor(state)
    move  = processor(blank)
    state = actuator(state, blank, move)
    step += 1
    print(f"Buoc {step}: {move}")
    display(state)

    if state == GOAL:
        print(f"Hoan thanh sau {step} buoc")
        break
    key = str(state)
    lichsu[key] = lichsu.get(key, 0) + 1
    if lichsu[key] >= 30:
        print("Dung! Trang thai lap qua nhieu lan.")
        break

In [ ]:
import random
def sensor(phong, pos):
    r, c = pos
    return phong[r][c]

def tim_o_ban_gan_nhat(phong, pos):
    r, c = pos
    tot_nhat = None
    kc_min = float("inf")
    for i in range(3):
        for j in range(3):
            if phong[i][j] == 1:
                kc = abs(i - r) + abs(j - c)
                if kc < kc_min:
                    kc_min = kc
                    tot_nhat = (i, j)
    return tot_nhat

def processor(tinh_trang, pos, phong):
    r, c = pos

    if tinh_trang == 1:
        return "HUT"
    dich = tim_o_ban_gan_nhat(phong, pos)
    if dich is None:
        return "DUNG"
    dr, dc = dich[0] - r, dich[1] - c
    huong_doc = None
    huong_ngang = None

    if dr < 0 and r - 1 >= 0 and phong[r-1][c] != 2:
        huong_doc = "LEN"
    elif dr > 0 and r + 1 <= 2 and phong[r+1][c] != 2:
        huong_doc = "XUONG"

    if dc < 0 and c - 1 >= 0 and phong[r][c-1] != 2:
        huong_ngang = "TRAI"
    elif dc > 0 and c + 1 <= 2 and phong[r][c+1] != 2:
        huong_ngang = "PHAI"
    if abs(dr) >= abs(dc):
        if huong_doc:
            return huong_doc
        elif huong_ngang:
            return huong_ngang
    else:
        if huong_ngang:
            return huong_ngang
        elif huong_doc:
            return huong_doc
    for h, (nr, nc) in [("LEN",(r-1,c)),("XUONG",(r+1,c)),("TRAI",(r,c-1)),("PHAI",(r,c+1))]:
        if 0 <= nr <= 2 and 0 <= nc <= 2 and phong[nr][nc] != 2:
            return h
    return "DUNG"

def actuator(hanh_dong, pos, phong):
    r, c = pos

    if hanh_dong == "HUT":
        phong[r][c] = 0
        print(f"-> HUT bui tai ({r},{c})")
    elif hanh_dong == "LEN":
        pos = (r - 1, c)
        print(f"-> Di LEN den ({r-1},{c})")
    elif hanh_dong == "XUONG":
        pos = (r + 1, c)
        print(f"-> Di XUONG den ({r+1},{c})")
    elif hanh_dong == "TRAI":
        pos = (r, c - 1)
        print(f"-> Di TRAI den ({r},{c-1})")
    elif hanh_dong == "PHAI":
        pos = (r, c + 1)
        print(f"-> Di PHAI den ({r},{c+1})")
    elif hanh_dong == "DUNG":
        print(f"-> DUNG tai ({r},{c})")

    return pos, phong

def con_ban(phong):
    for hang in phong:
        for o in hang:
            if o == 1:
                return True
    return False

def main():
    print("Nhap phong 3x3")

    phong = []
    for i in range(3):
        while True:
            raw = input(f"Hang {i+1}: ").split()
            if len(raw) == 3 and all(x in "012" for x in raw):
                phong.append(list(map(int, raw)))
                break
            print("Nhap lai")

    while True:
        raw = input("Vi tri robot: ").split()
        if len(raw) == 2:
            r, c = int(raw[0]), int(raw[1])
            if 0 <= r <= 2 and 0 <= c <= 2 and phong[r][c] != 2:
                pos = (r, c)
                break
        print("Vi tri khong hop le, thu lai")

    print()
    buoc = 0
    while con_ban(phong):
        buoc += 1
        print(f"Buoc {buoc}")
        tinh_trang = sensor(phong, pos)
        hanh_dong  = processor(tinh_trang, pos, phong)

        if hanh_dong == "DUNG":
            print(" khong the tiep tuc")
            break

        pos, phong = actuator(hanh_dong, pos, phong)
    if not con_ban(phong):
        print(f"Phong da sach sau {buoc} buoc")
if __name__ == "__main__":
    main()

In [ ]:
import random
GOAL = [[1,2,3],[4,5,6],[7,8,0]]
model = {
    "trang_thai_hien_tai": None,  
    "da_tham":             set(), 
    "lich_su":             [],   
}
def hien_thi(state):
    for hang in state:
        print(f"Da tham: {len(model['da_tham'])} trang thai  |  Lich su: {model['lich_su'][-5:]}")

def to_tuple(state):
    return tuple(tuple(h) for h in state)

def update_model(state, huong):
    model["trang_thai_hien_tai"] = state
    model["da_tham"].add(to_tuple(state))
    if huong:
        model["lich_su"].append(huong)
def sensor(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return (i, j)
def processor(state, blank):
    r, c = blank
    ung_vien = []
    if r > 0:
        moi = [h[:] for h in state]
        moi[r][c], moi[r-1][c] = moi[r-1][c], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("LEN", moi))
    if r < 2:
        moi = [h[:] for h in state]
        moi[r][c], moi[r+1][c] = moi[r+1][c], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("XUONG", moi))
    if c > 0:
        moi = [h[:] for h in state]
        moi[r][c], moi[r][c-1] = moi[r][c-1], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("TRAI", moi))
    if c < 2:
        moi = [h[:] for h in state]
        moi[r][c], moi[r][c+1] = moi[r][c+1], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("PHAI", moi))
    if not ung_vien:
        return None, None 
    return random.choice(ung_vien)
def actuator(huong, state):
    print(f" Di chuyen: {huong}")
    return state
def main():
    print("Nhap ma tran 3x3:")
    state = []
    for i in range(3):
        while True:
            raw = input(f"Hang {i+1}: ").split()
            if len(raw) == 3:
                state.append(list(map(int, raw)))
                break
            print("Nhap lai")
    print("Trang thai ban dau:")
    hien_thi(state)
    update_model(state, None)
    buoc = 0
    while state != GOAL:
        buoc += 1
        print(f" Buoc {buoc}")
        blank= sensor(state)
        huong, state = processor(state, blank)
        if huong is None:
            print("Bi ket, khong con trang thai moi!")
            break

        state = actuator(huong, state)
        update_model(state, huong)
        hien_thi(state)

        if buoc > 5000:
            print("Qua nhieu buoc, dung lai.")
            break

    if state == GOAL:
        print(f" Da giai xong sau {buoc} buoc")
        print(f"Toan bo lich su ({len(model['lich_su'])} buoc): {model['lich_su']}")

if __name__ == "__main__":
    main()

In [ ]:

model = {
    "trang_thai_hien_tai": None,  
    "da_tham":             set(), 
    "lich_su":             [],    
    "o_ban_con_lai":       [],   
}

def hien_thi(phong, pos):
    r, c = pos
    for i in range(3):
        hang = "|"
        for j in range(3):
            if (i, j) == (r, c):
                hang += " R |"
            elif phong[i][j] == 0:
                hang += "   |"
            elif phong[i][j] == 1:
                hang += " * |"
        print(hang)
    print(f"Da tham: {sorted(model['da_tham'])}  |  Con ban: {model['o_ban_con_lai']}")

def update_model(phong, pos, hanh_dong):
    model["trang_thai_hien_tai"] = (phong, pos)
    model["da_tham"].add(pos)
    if hanh_dong:
        model["lich_su"].append(hanh_dong)
    model["o_ban_con_lai"] = [
        (i, j)
        for i in range(3)
        for j in range(3)
        if phong[i][j] == 1
    ]

def sensor(phong, pos):
    r, c = pos
    return phong[r][c]

def processor(tinh_trang, pos, phong):
    r, c = pos
    if tinh_trang == 1:
        return "HUT"
    if not model["o_ban_con_lai"]:
        return "DUNG"
    dich = None
    kc_min = float("inf")
    for (i, j) in model["o_ban_con_lai"]:
        kc = abs(i - r) + abs(j - c)
        if kc < kc_min:
            kc_min = kc
            dich = (i, j)
    dr, dc = dich[0] - r, dich[1] - c

    if abs(dr) >= abs(dc):
        if dr < 0 and r - 1 >= 0 and (r-1, c) not in model["da_tham"]:
            return "LEN"
        if dr > 0 and r + 1 <= 2 and (r+1, c) not in model["da_tham"]:
            return "XUONG"
        if dc < 0 and c - 1 >= 0 and (r, c-1) not in model["da_tham"]:
            return "TRAI"
        if dc > 0 and c + 1 <= 2 and (r, c+1) not in model["da_tham"]:
            return "PHAI"
    else:
        if dc < 0 and c - 1 >= 0 and (r, c-1) not in model["da_tham"]:
            return "TRAI"
        if dc > 0 and c + 1 <= 2 and (r, c+1) not in model["da_tham"]:
            return "PHAI"
        if dr < 0 and r - 1 >= 0 and (r-1, c) not in model["da_tham"]:
            return "LEN"
        if dr > 0 and r + 1 <= 2 and (r+1, c) not in model["da_tham"]:
            return "XUONG"
    if r > 0: return "LEN"
    if r < 2: return "XUONG"
    if c > 0: return "TRAI"
    if c < 2: return "PHAI"
    return "DUNG"

def actuator(hanh_dong, pos, phong):
    r, c = pos

    if hanh_dong == "HUT":
        phong[r][c] = 0
        print(f"HUT bui tai ({r},{c})")
    elif hanh_dong == "LEN":
        pos = (r - 1, c)
        print(f" Di LEN den ({r-1},{c})")
    elif hanh_dong == "XUONG":
        pos = (r + 1, c)
        print(f"Di XUONG den ({r+1},{c})")
    elif hanh_dong == "TRAI":
        pos = (r, c - 1)
        print(f" Di TRAI den ({r},{c-1})")
    elif hanh_dong == "PHAI":
        pos = (r, c + 1)
        print(f" Di PHAI den ({r},{c+1})")
    elif hanh_dong == "DUNG":
        print(f" DUNG tai ({r},{c})")
    return pos, phong
def main():
    print("Nhap phong 3x3:")
    phong = []
    for i in range(3):
        while True:
            raw = input(f"Hang {i+1}: ").split()
            if len(raw) == 3 and all(x in "01" for x in raw):
                phong.append(list(map(int, raw)))
                break
            print("Nhap lai")

    while True:
        raw = input("Vi tri robot: ").split()
        if len(raw) == 2:
            r, c = int(raw[0]), int(raw[1])
            if 0 <= r <= 2 and 0 <= c <= 2:
                pos = (r, c)
                break
        print("Vi tri khong hop le")

    print()
    update_model(phong, pos, None)
    hien_thi(phong, pos)

    buoc = 0
    while model["o_ban_con_lai"]:
        buoc += 1
        tinh_trang = sensor(phong, pos)
        hanh_dong  = processor(tinh_trang, pos, phong)
        if hanh_dong == "DUNG":
            print(" DUNG, khong the tiep tuc.")
            break
        pos, phong = actuator(hanh_dong, pos, phong)
        update_model(phong, pos, hanh_dong)
        hien_thi(phong, pos)
    if not model["o_ban_con_lai"]:
        print(f"Phong da sach sau {buoc} buoc")
        print(f"Lich su: {model['lich_su']}")
if __name__ == "__main__":
    main()